**⚠️⚠️ 首次使用，建议先看最底部使用说明**

In [ ]:
# @title 💾 挂载Google硬盘
%%capture
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title ⚗️ z-image-base-INT8量化 （连接CPU,只需运行一次，今后使用不再运行）

# ==========================================
# 步骤 1：安装依赖
# ==========================================
!pip install -U -q git+https://github.com/Disty0/sdnq git+https://github.com/huggingface/diffusers
!pip install -U -q transformers accelerate safetensors psutil

# ==========================================
# 步骤 2：主量化流程
# ==========================================
import os
import json
import gc
import shutil
import torch
from google.colab import drive
from safetensors.torch import save_file, load_file
from huggingface_hub import hf_hub_download
import psutil

def print_mem():
    mem = psutil.virtual_memory()
    used_gb = mem.used / 1e9
    total_gb = mem.total / 1e9
    percent = mem.percent
    print(f"💻 RAM: {used_gb:.1f}GB / {total_gb:.1f}GB ({percent:.1f}%)")
    if percent > 85:
        print("⚠️  内存使用超过 85%，强制 GC")
        gc.collect()

def force_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# 挂载 Drive
if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

FINAL_DIR = "/content/drive/MyDrive/z-image-base-transformer-int8"
LOCAL_DIR = "/content/quant_output"
TEMP_DIR = "/content/temp_quant"

for d in [LOCAL_DIR, TEMP_DIR]:
    shutil.rmtree(d, ignore_errors=True)
    os.makedirs(d, exist_ok=True)

# 与 Turbo 完全一致的排除列表
target_modules_to_not_convert = [
    "layers.0.adaLN_modulation.0.weight",
    "all_x_embedder",
    "t_embedder",
    "cap_embedder",
    "all_final_layer"
]

def should_quantize(name):
    """判断是否量化（与 Turbo 逻辑一致）"""
    if name in target_modules_to_not_convert:
        return False

    for skip_prefix in target_modules_to_not_convert:
        if name.startswith(skip_prefix):
            return False

    return name.endswith(".weight")

def quantize_tensor_int8_per_channel(tensor):
    """
    INT8 Per-Channel 对称量化（与 Turbo 完全对齐）

    对于 2D 权重 [out_features, in_features]：
    - 对每个输出通道（dim=0）独立计算 scale
    - 返回 scale shape: [out_features, 1]（与 Turbo 一致）

    对于 1D 权重（如 bias/norm）：
    - 使用 per-tensor 量化
    """
    if tensor.dtype == torch.int8:
        return tensor, None

    t_float = tensor.float()

    # 🔧 关键：2D 权重用 per-channel，1D 用 per-tensor
    if tensor.dim() == 2:
        # Per-Channel 量化
        # 沿 dim=1 求最大值，保留 dim=0（输出通道）
        abs_max_per_channel = t_float.abs().amax(dim=1, keepdim=True)  # shape: [out_features, 1]

        # 避免除零
        abs_max_per_channel = torch.clamp(abs_max_per_channel, min=1e-8)

        # 计算 scale（每个输出通道一个）
        scale = abs_max_per_channel / 127.0  # shape: [out_features, 1]

        # 量化（自动广播）
        q_tensor = (t_float / scale).round().clamp(-128, 127).to(torch.int8)

        # 🔧 关键：保持 scale 的 shape 为 [out_features, 1]（与 Turbo 一致）
        return q_tensor, scale.to(torch.bfloat16)

    else:
        # Per-Tensor 量化（用于 1D/3D+ 权重）
        abs_max = t_float.abs().max().item()

        if abs_max == 0 or abs_max < 1e-8:
            return tensor.to(torch.int8), torch.tensor(1.0, dtype=torch.bfloat16)

        scale = abs_max / 127.0
        q_tensor = (t_float / scale).round().clamp(-128, 127).to(torch.int8)

        # 🔧 返回标量 scale
        return q_tensor, torch.tensor(scale, dtype=torch.bfloat16)

print("="*60)
print("🚀 开始 Per-Channel 量化流程（Turbo 完全对齐）")
print("="*60)
print_mem()

# ==========================================
# 1. 下载原始模型
# ==========================================
print("\n📥 步骤 1: 下载模型索引...")

index_file = hf_hub_download(
    "Tongyi-MAI/Z-Image",
    "transformer/diffusion_pytorch_model.safetensors.index.json",
    cache_dir=TEMP_DIR,
    resume_download=True
)

with open(index_file, "r") as f:
    original_index = json.load(f)

weight_map = original_index["weight_map"]
shard_files = sorted(set(weight_map.values()))

print(f"   ✓ 原始分片数: {len(shard_files)}")
print(f"   ✓ 总参数数: {len(weight_map)}")

config_file = hf_hub_download(
    "Tongyi-MAI/Z-Image",
    "transformer/config.json",
    cache_dir=TEMP_DIR,
    resume_download=True
)

with open(config_file, "r") as f:
    base_config = json.load(f)

print(f"   ✓ config.json 已下载")
print_mem()

# ==========================================
# 2. 逐分片量化
# ==========================================
print(f"\n🔄 步骤 2: 逐分片量化...")

TEMP_SHARD_SIZE = 500_000_000  # 500MB

temp_shard_idx = 1
current_temp_shard = {}
current_temp_size = 0
scales_dict = {}

quantized_count = 0
kept_count = 0

for shard_idx, shard_name in enumerate(shard_files):
    print(f"\n{'─'*60}")
    print(f"📦 分片 {shard_idx+1}/{len(shard_files)}: {shard_name}")
    print_mem()

    shard_path = hf_hub_download(
        "Tongyi-MAI/Z-Image",
        f"transformer/{shard_name}",
        cache_dir=TEMP_DIR,
        resume_download=True
    )

    shard_data = load_file(shard_path)
    print(f"   包含参数: {len(shard_data)} 个")

    for param_idx, (param_name, tensor) in enumerate(shard_data.items()):
        tensor = tensor.cpu()

        if should_quantize(param_name) and tensor.dim() >= 2:
            # 🔧 使用 Per-Channel 量化
            q_tensor, scale = quantize_tensor_int8_per_channel(tensor)

            if scale is not None:
                # 🔧 关键：scale 键名格式
                scale_key = param_name.replace(".weight", ".scale")
                scales_dict[scale_key] = scale

                quantized_count += 1

                if param_idx < 3:
                    print(f"   ✓ [{param_idx+1}] {param_name}")
                    print(f"      量化: {list(tensor.shape)} → int8")
                    print(f"      scale shape: {list(scale.shape)} (per-channel)")

        else:
            # 保持为 bfloat16
            q_tensor = tensor.to(torch.bfloat16) if tensor.dtype != torch.bfloat16 else tensor
            kept_count += 1

            if param_idx < 3:
                print(f"   - [{param_idx+1}] {param_name}")
                print(f"      保持: bfloat16")

        param_size = q_tensor.numel() * q_tensor.element_size()

        # 内存控制：达到临时分片大小就保存
        if current_temp_size + param_size > TEMP_SHARD_SIZE and current_temp_shard:
            temp_file = os.path.join(TEMP_DIR, f"temp_{temp_shard_idx:04d}.safetensors")
            print(f"\n   💾 暂存临时分片 #{temp_shard_idx} ({len(current_temp_shard)} 参数)")
            save_file(current_temp_shard, temp_file)

            current_temp_shard.clear()
            current_temp_size = 0
            temp_shard_idx += 1
            force_cleanup()

        current_temp_shard[param_name] = q_tensor
        current_temp_size += param_size

        del tensor, q_tensor

    del shard_data
    force_cleanup()

    try:
        os.remove(shard_path)
    except:
        pass

# 保存最后一个临时分片
if current_temp_shard:
    temp_file = os.path.join(TEMP_DIR, f"temp_{temp_shard_idx:04d}.safetensors")
    print(f"\n💾 保存最后临时分片 #{temp_shard_idx}")
    save_file(current_temp_shard, temp_file)
    del current_temp_shard
    force_cleanup()

total_temp_shards = temp_shard_idx

# 保存 scales
if scales_dict:
    print(f"\n💾 保存量化 scales: {len(scales_dict)} 个")

    # 🔧 验证：检查 scale 的 shape
    sample_scale_keys = list(scales_dict.keys())[:3]
    print(f"   前 3 个 scale 的 shape:")
    for k in sample_scale_keys:
        print(f"      {k}: {list(scales_dict[k].shape)}")

    save_file(scales_dict, os.path.join(TEMP_DIR, "temp_scales.safetensors"))
    del scales_dict
    force_cleanup()

print(f"\n{'='*60}")
print(f"✅ 量化统计:")
print(f"   量化参数: {quantized_count}")
print(f"   保持参数: {kept_count}")
print(f"   临时分片数: {total_temp_shards}")
print(f"{'='*60}")
print_mem()

# ==========================================
# 3. 合并为单个文件
# ==========================================
print(f"\n🔗 步骤 3: 合并为单个文件...")

final_state_dict = {}

for i in range(1, total_temp_shards + 1):
    temp_file = os.path.join(TEMP_DIR, f"temp_{i:04d}.safetensors")

    if not os.path.exists(temp_file):
        continue

    print(f"   [{i}/{total_temp_shards}] 加载临时分片...")
    temp_data = load_file(temp_file)
    final_state_dict.update(temp_data)
    del temp_data
    force_cleanup()

    if i % 5 == 0:
        print_mem()

# 加载 scales
scales_file = os.path.join(TEMP_DIR, "temp_scales.safetensors")
if os.path.exists(scales_file):
    print(f"   加载 scales...")
    scales_data = load_file(scales_file)
    final_state_dict.update(scales_data)
    del scales_data
    force_cleanup()

print(f"\n✅ 合并统计:")
print(f"   总参数数: {len(final_state_dict)}")

# 验证
scale_keys = [k for k in final_state_dict.keys() if k.endswith('.scale')]
weight_keys = [k for k in final_state_dict.keys() if k.endswith('.weight')]

print(f"   权重参数: {len(weight_keys)}")
print(f"   Scale 参数: {len(scale_keys)}")

# 🔧 验证 scale 的 shape
print(f"\n   验证 scale shape（前 3 个）:")
for k in sorted(scale_keys)[:3]:
    scale = final_state_dict[k]
    print(f"      {k}: {list(scale.shape)}")

# 保存单文件
print(f"\n💾 保存最终文件...")

save_file(
    final_state_dict,
    os.path.join(LOCAL_DIR, "diffusion_pytorch_model.safetensors"),
    metadata={"format": "pt"}
)

file_size = os.path.getsize(os.path.join(LOCAL_DIR, "diffusion_pytorch_model.safetensors"))
print(f"   ✓ 文件大小: {file_size/1e9:.2f} GB")

del final_state_dict
force_cleanup()
print_mem()

# ==========================================
# 4. 生成配置文件
# ==========================================
print(f"\n📝 步骤 4: 生成配置文件...")

quantization_config_standalone = {
    "add_skip_keys": False,
    "dequantize_fp32": False,
    "group_size": -1,
    "is_integer": True,
    "is_training": False,
    "modules_dtype_dict": {},
    "modules_to_not_convert": target_modules_to_not_convert,
    "non_blocking": False,
    "quant_conv": False,
    "quant_method": "sdnq",
    "quantization_device": None,
    "quantized_matmul_dtype": None,
    "return_device": None,
    "sdnq_version": "0.1.2",
    "svd_rank": 32,
    "svd_steps": 8,
    "use_grad_ckpt": True,
    "use_quantized_matmul": False,
    "use_quantized_matmul_conv": False,
    "use_static_quantization": True,
    "use_stochastic_rounding": False,
    "use_svd": False,
    "weights_dtype": "int8"
}

with open(os.path.join(LOCAL_DIR, "quantization_config.json"), "w") as f:
    json.dump(quantization_config_standalone, f, indent=2)

quantization_config_embedded = {
    **quantization_config_standalone,
    "quantization_device": "xpu",
    "return_device": "cpu",
    "add_skip_keys": True,
}

base_config["quantization_config"] = quantization_config_embedded
base_config["_diffusers_version"] = "0.36.0.dev0"
base_config.pop("siglip_feat_dim", None)

with open(os.path.join(LOCAL_DIR, "config.json"), "w") as f:
    json.dump(base_config, f, indent=2)

print("   ✓ 配置文件已生成")

# ==========================================
# 5. 拷贝到 Drive
# ==========================================
print(f"\n📤 步骤 5: 拷贝到 Drive...")
print(f"   目标: {FINAL_DIR}")

if os.path.exists(FINAL_DIR):
    shutil.rmtree(FINAL_DIR)

shutil.copytree(LOCAL_DIR, FINAL_DIR)
print("   ✓ 拷贝完成")

print(f"\n🧹 清理临时文件...")
shutil.rmtree(LOCAL_DIR, ignore_errors=True)
shutil.rmtree(TEMP_DIR, ignore_errors=True)
force_cleanup()

# ==========================================
# 6. 最终验证
# ==========================================
print(f"\n{'='*60}")
print(f"🎉 Per-Channel 量化完成！")
print(f"{'='*60}")

print(f"\n📁 最终文件:")
for f in sorted(os.listdir(FINAL_DIR)):
    fpath = os.path.join(FINAL_DIR, f)
    if os.path.isfile(fpath):
        fsize = os.path.getsize(fpath)
        if fsize > 1e6:
            print(f"   ✓ {f:<45} {fsize/1e9:>8.2f} GB")
        else:
            print(f"   ✓ {f:<45} {fsize/1e3:>8.1f} KB")

print(f"\n✅ 与 Turbo 对齐检查:")
print(f"   ✓ 量化方式: Per-Channel")
print(f"   ✓ Scale shape: [out_features, 1]")
print(f"   ✓ Scale dtype: bfloat16")
print(f"   ✓ 未量化层: bfloat16")
print(f"   ✓ 量化参数数: {quantized_count}")

print(f"\n🚀 使用方法:")
print(f"   !cp -rf {FINAL_DIR}/* \\")
print(f"           /content/Z-Image-Turbo-SDNQ-int8/transformer/")

print_mem()

from IPython.display import clear_output

final_message = "量化模型已保存于google driver，今后无需再运行，请重启系统，连接GPU，跳过本单元格，重新运行程序"

clear_output(wait=True)
print("✅ " + final_message)

In [ ]:
# @title 🗑️ 控制显存碎片配置
import os, sys

# 必须在 import torch / diffusers / transformers 之前运行
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

# 防呆：如果 torch 已经被导入了，就说明这格跑晚了（本次不会生效）
if "torch" in sys.modules:
    raise RuntimeError("torch 已导入：allocator 配置本次不会生效。请重启 runtime 后第一格先跑这个 cell。")

print("✅ PYTORCH_CUDA_ALLOC_CONF =", os.environ["PYTORCH_CUDA_ALLOC_CONF"])

In [ ]:
# @title 🎛️环境配置
%%capture
!pip install -U --prefer-binary -q \
  git+https://github.com/Disty0/sdnq \
  git+https://github.com/huggingface/diffusers

!pip install -U -q transformers accelerate

In [ ]:
# @title 🤖 模型管理器（五合一）
import ipywidgets as widgets
from IPython.display import display
import torch
import os
import gc
import weakref
import shutil

# ----------------------------
# ✅ TF32 精度控制
# ----------------------------
try:
    torch.set_float32_matmul_precision('high')
    torch.backends.cudnn.conv.fp32_precision = 'tf32'
except AttributeError:
    pass

# ----------------------------
# 路径常量
# ----------------------------
PATH_INT8      = "/content/Z-Image-Turbo-SDNQ-int8"
PATH_UINT4     = "/content/Z-Image-Turbo-SDNQ-uint4"
PATH_BASE_INT8 = "/content/drive/MyDrive/z-image-base-transformer-int8"
PATH_BASE_INT8_CACHE = "/content/z-image-base-transformer-int8-cache"

# ----------------------------
# 所有文件清单（用于下载/检查）
# ----------------------------
FILES_COMMON =[
    "model_index.json",
    "scheduler/scheduler_config.json",
    "tokenizer/merges.txt",
    "tokenizer/tokenizer.json",
    "tokenizer/tokenizer_config.json",
    "tokenizer/vocab.json",
    "vae/config.json",
    "vae/diffusion_pytorch_model.safetensors",
]
FILES_TE =[
    "text_encoder/config.json",
    "text_encoder/generation_config.json",
    "text_encoder/model.safetensors",
    "text_encoder/quantization_config.json",
]
FILES_TRANSFORMER_INT8 =[
    "transformer/config.json",
    "transformer/diffusion_pytorch_model.safetensors",
    "transformer/quantization_config.json",
]
FILES_TRANSFORMER_UINT4 =[
    "transformer/config.json",
    "transformer/diffusion_pytorch_model.safetensors",
    "transformer/quantization_config.json",
]

# ----------------------------
# 五种模型配置
# ----------------------------
MODEL_CATALOG = {
    "⚡ uint4（预览/更快）": {
        "main_path"  : PATH_UINT4,
        "main_repo"  : "Disty0/Z-Image-Turbo-SDNQ-uint4-svd-r32",
        "main_files" : FILES_COMMON + FILES_TE + FILES_TRANSFORMER_UINT4,
        "te_path"    : None,
        "te_repo"    : None,
        "te_files"   :[],
        "transformer_path"      : None,
        "transformer_is_gdrive" : False,
        "description": "完整 uint4 套件，速度最快，适合预览出图",
    },
    "🎯 int8（产品/更好）": {
        "main_path"  : PATH_INT8,
        "main_repo"  : "Disty0/Z-Image-Turbo-SDNQ-int8",
        "main_files" : FILES_COMMON + FILES_TE + FILES_TRANSFORMER_INT8,
        "te_path"    : None,
        "te_repo"    : None,
        "te_files"   :[],
        "transformer_path"      : None,
        "transformer_is_gdrive" : False,
        "description": "完整 int8 套件，画质最佳，适合产品出图",
    },
    "🔀 混搭①：uint4-TE + int8 主体": {
        "main_path"  : PATH_INT8,
        "main_repo"  : "Disty0/Z-Image-Turbo-SDNQ-int8",
        "main_files" : FILES_COMMON + FILES_TRANSFORMER_INT8,
        "te_path"    : PATH_UINT4,
        "te_repo"    : "Disty0/Z-Image-Turbo-SDNQ-uint4-svd-r32",
        "te_files"   : FILES_TE,
        "transformer_path"      : None,
        "transformer_is_gdrive" : False,
        "description": "int8 骨架 + uint4 TE，兼顾速度与画质",
    },
    "🔩 混搭②：uint4 + base-int8 Transformer（GDrive）": {
        "main_path"  : PATH_UINT4,
        "main_repo"  : "Disty0/Z-Image-Turbo-SDNQ-uint4-svd-r32",
        "main_files" : FILES_COMMON + FILES_TE,
        "te_path"    : None,
        "te_repo"    : None,
        "te_files"   :[],
        "transformer_path"      : PATH_BASE_INT8,
        "transformer_is_gdrive" : True,
        "description": "uint4 骨架 + Google Drive base-int8 Transformer，直读 GDrive 无需下载",
    },
    "🔧 混搭③：int8 + base-int8 Transformer（GDrive）": {
        "main_path"  : PATH_INT8,
        "main_repo"  : "Disty0/Z-Image-Turbo-SDNQ-int8",
        "main_files" : FILES_COMMON + FILES_TE,
        "te_path"    : None,
        "te_repo"    : None,
        "te_files"   :[],
        "transformer_path"      : PATH_BASE_INT8,
        "transformer_is_gdrive" : True,
        "description": "int8 骨架 + Google Drive base-int8 Transformer，直读 GDrive 无需下载",
    },
}

# ----------------------------
# UI 组件
# ----------------------------
output = widgets.Output()

model_dropdown = widgets.Dropdown(
    options=list(MODEL_CATALOG.keys()),
    value="⚡ uint4（预览/更快）",
    description="选择方案:",
    style={'description_width': '70px'},
    layout=widgets.Layout(width="98%"),
)

desc_html   = widgets.HTML(value="")
repo_html   = widgets.HTML(value="")
status_html = widgets.HTML(value="<span style='color:gray;'>准备就绪</span>")
progress_html = widgets.HTML(value="")
local_status_html = widgets.HTML(value="")

path_input = widgets.Text(
    value=MODEL_CATALOG[model_dropdown.value]["main_path"],
    placeholder="主体本地路径",
    description="主体路径:",
    style={'description_width': '70px'},
    layout=widgets.Layout(width="98%")
)

vae_path_input = widgets.Text(
    value="",
    placeholder="（可选）留空=模型自带 VAE",
    description="VAE 路径:",
    style={'description_width': '70px'},
    layout=widgets.Layout(width="98%")
)
vae_status_html = widgets.HTML(value="<span style='color:gray;'>VAE：默认（使用模型自带）</span>")

mode_dropdown = widgets.Dropdown(
    options=[
        ("📁 从本地路径加载", "local"),
        ("⬇️ 下载/补全所需文件", "download"),
        ("🔧 检查并修复缺失文件", "repair"),
    ],
    value="local",
    description="操作模式:",
    style={'description_width': '70px'},
    layout=widgets.Layout(width="98%")
)

execute_btn = widgets.Button(
    description="🚀 执行",
    button_style="primary",
    layout=widgets.Layout(width="120px", height="40px")
)
cleanup_btn = widgets.Button(
    description="🧹 清除旧模型",
    button_style="warning",
    layout=widgets.Layout(width="145px", height="40px")
)

cache_btn = widgets.Button(
    description="💾 缓存到本地",
    button_style="info",
    layout=widgets.Layout(width="145px", height="40px"),
    tooltip=f"将 GDrive Transformer 复制到 {PATH_BASE_INT8_CACHE}，下次加载自动优先使用本地缓存",
)
cache_status_html = widgets.HTML(value="")

def _gdrive_cache_valid():
    needed =["config.json", "diffusion_pytorch_model.safetensors", "quantization_config.json"]
    return all(
        os.path.exists(os.path.join(PATH_BASE_INT8_CACHE, f)) and
        os.path.getsize(os.path.join(PATH_BASE_INT8_CACHE, f)) > 0
        for f in needed
    )

def refresh_cache_status():
    if not is_gdrive_transformer():
        cache_status_html.value = ""
        return
    if _gdrive_cache_valid():
        cache_status_html.value = (
            f"<span style='color:green;'>✅ 本地缓存有效，加载时将自动使用 "
            f"<code>{PATH_BASE_INT8_CACHE}</code></span>"
        )
    else:
        cache_status_html.value = (
            f"<span style='color:orange;'>⚠️ 本地无缓存，将直读 GDrive（较慢）。"
            f"可点击「💾 缓存到本地」加速后续加载。</span>"
        )

def cfg(): return MODEL_CATALOG[model_dropdown.value]
def is_gdrive_transformer(): return cfg().get("transformer_is_gdrive", False)
def has_external_te(): return cfg().get("te_path") is not None

def update_info():
    c = cfg()
    desc_html.value = f"""
    <div style='background:#1e293b;color:#e2e8f0;padding:8px 12px;border-radius:6px;font-size:12px;margin-top:4px;'>
        <b style='color:#38bdf8;'>💡 {model_dropdown.value}</b><br>
        {c['description']}
    </div>"""

    lines =[f"<b>主体 Repo:</b> <code>{c['main_repo']}</code>"]
    if c.get("te_path"):
        lines.append(f"<b>TE Repo:</b> <code>{c['te_repo']}</code> → <code>{c['te_path']}/text_encoder/</code>")
    if c.get("transformer_path"):
        src = "☁️ Google Drive（直读，不下载）" if c["transformer_is_gdrive"] else c["transformer_path"]
        lines.append(f"<b>Transformer:</b> {src}")
    repo_html.value = "<div style='font-size:12px;color:#64748b;margin:4px 0;'>" + "<br>".join(lines) + "</div>"

def refresh_local_status():
    c = cfg()
    main = path_input.value.strip()
    msgs = []
    missing_main =[f for f in c["main_files"] if not _file_ok(main, f)]
    if missing_main: msgs.append(f"主体缺失 {len(missing_main)} 个文件")
    if c.get("te_path"):
        missing_te = [f for f in c["te_files"] if not _file_ok(c["te_path"], f)]
        if missing_te: msgs.append(f"TE 缺失 {len(missing_te)} 个文件")
    if c.get("transformer_path") and c["transformer_is_gdrive"]:
        tp = c["transformer_path"]
        tw = os.path.join(tp, "diffusion_pytorch_model.safetensors")
        if not os.path.exists(tw): msgs.append(f"⚠️ GDrive transformer 未找到: {tp}")

    if msgs: local_status_html.value = f"<span style='color:orange;'>⚠️ {' | '.join(msgs)}</span>"
    else: local_status_html.value = "<span style='color:green;'>✅ 所需文件完整，可直接加载</span>"

def _file_ok(base, rel):
    p = os.path.join(base, rel)
    return os.path.exists(p) and os.path.getsize(p) > 0

def refresh_vae_status():
    p = (vae_path_input.value or "").strip()
    if not p:
        vae_status_html.value = "<span style='color:gray;'>VAE：默认（使用模型自带）</span>"
        return
    try:
        vd = _resolve_vae_dir(p)
        vae_status_html.value = (f"<span style='color:green;'>✅ 外置 VAE: {vd}</span>")
    except Exception as e:
        vae_status_html.value = f"<span style='color:red;'>❌ VAE 路径无效：{e}</span>"

def on_model_change(change):
    path_input.value = cfg()["main_path"]
    update_info()
    refresh_local_status()
    refresh_cache_status()

model_dropdown.observe(on_model_change, names="value")
path_input.observe(lambda c: refresh_local_status(), names="value")
vae_path_input.observe(lambda c: refresh_vae_status(), names="value")
update_info()
refresh_local_status()
refresh_vae_status()
refresh_cache_status()

def cache_transformer_to_local():
    src = PATH_BASE_INT8
    dst = PATH_BASE_INT8_CACHE
    needed =["config.json", "diffusion_pytorch_model.safetensors", "quantization_config.json"]

    if not os.path.isdir(src):
        raise RuntimeError(f"GDrive 源路径不存在或未挂载：{src}")

    os.makedirs(dst, exist_ok=True)
    for fname in needed:
        src_f = os.path.join(src, fname)
        dst_f = os.path.join(dst, fname)
        if not os.path.exists(src_f):
            raise RuntimeError(f"GDrive 中缺少文件：{src_f}")
        if os.path.exists(dst_f) and os.path.getsize(dst_f) == os.path.getsize(src_f):
            print(f"⏭️ 已缓存，跳过：{fname}")
            continue
        sz_mb = os.path.getsize(src_f) / 1024**2
        print(f"📋 复制 {fname}（{sz_mb:.1f} MB）...")
        shutil.copy2(src_f, dst_f)
        print(f"   ✅ 完成")

def on_cache_click(b):
    if not is_gdrive_transformer():
        with output:
            print("ℹ️ 当前方案不含 GDrive Transformer，无需缓存。")
        return
    output.clear_output()
    cache_btn.disabled = True
    cache_status_html.value = "<span style='color:orange;'>⏳ 正在复制到本地...</span>"
    with output:
        try:
            cache_transformer_to_local()
            print(f"\n✅ 缓存完成！路径：{PATH_BASE_INT8_CACHE}")
            print("下次执行加载时将自动优先使用本地缓存，无需再次操作。")
            cache_status_html.value = (
                f"<span style='color:green;'>✅ 缓存完成 → <code>{PATH_BASE_INT8_CACHE}</code></span>"
            )
        except Exception as e:
            print(f"❌ 缓存失败：{e}")
            cache_status_html.value = f"<span style='color:red;'>❌ 缓存失败：{e}</span>"
        finally:
            cache_btn.disabled = False
            refresh_cache_status()

cache_btn.on_click(on_cache_click)

def download_file(repo_id, filename, save_path):
    from huggingface_hub import hf_hub_download
    try:
        hf_hub_download(
            repo_id=repo_id, filename=filename,
            local_dir=save_path, local_dir_use_symlinks=False, force_download=True,
        )
        lf = os.path.join(save_path, filename)
        if os.path.exists(lf) and os.path.getsize(lf) > 0: return True, os.path.getsize(lf)
        return False, 0
    except Exception as e:
        return False, str(e)

def download_filelist(repo_id, base_path, file_list, label=""):
    os.makedirs(base_path, exist_ok=True)
    results = {"success": 0, "failed": 0, "skipped": 0}
    failed =[]
    for i, f in enumerate(file_list):
        progress_html.value = f"<span style='color:blue;'>📥[{label}{i+1}/{len(file_list)}] {f}</span>"
        if _file_ok(base_path, f):
            sz = os.path.getsize(os.path.join(base_path, f)) / 1024**2
            print(f"⏭️ 已存在：{f} ({sz:.1f} MB)")
            results["skipped"] += 1
            continue
        print(f"⬇️ 下载：{f}...")
        ok, info = download_file(repo_id, f, base_path)
        if ok:
            print(f"   ✅ {info/1024**2:.1f} MB")
            results["success"] += 1
        else:
            print(f"   ❌ {info}")
            results["failed"] += 1
            failed.append(f)
    progress_html.value = ""
    return results, failed

def do_download():
    c = cfg()
    main_path = path_input.value.strip()
    total = {"success":0, "failed":0, "skipped":0}
    all_failed =[]

    print(f"📦 [主体] {c['main_repo']} → {main_path}")
    print(f"   文件数：{len(c['main_files'])}")
    print("-" * 55)
    r, f = download_filelist(c["main_repo"], main_path, c["main_files"], "主体 ")
    for k in total: total[k] += r[k]
    all_failed += f

    if c.get("te_path") and c.get("te_repo"):
        print(f"\n📦 [TE] {c['te_repo']} → {c['te_path']}")
        print(f"   文件数：{len(c['te_files'])}")
        print("-" * 55)
        r2, f2 = download_filelist(c["te_repo"], c["te_path"], c["te_files"], "TE ")
        for k in total: total[k] += r2[k]
        all_failed += f2

    if c.get("transformer_path") and c["transformer_is_gdrive"]:
        tp = c["transformer_path"]
        tw = os.path.join(tp, "diffusion_pytorch_model.safetensors")
        print(f"\n☁️[GDrive Transformer] 路径：{tp}")
        if os.path.exists(tw): print(f"   ✅ safetensors 已存在，直读即用")
        else: print(f"   ⚠️  文件不存在！请确认 Google Drive 已挂载且路径正确。")

    print("=" * 55)
    print(f"📊 下载 {total['success']}  跳过 {total['skipped']}  失败 {total['failed']}")
    if all_failed:
        print("❌ 失败文件：")
        for f in all_failed: print(f"   - {f}")
    return total, all_failed

def do_repair():
    c = cfg()
    main_path = path_input.value.strip()
    total_fixed, total_needed = 0, 0
    any_fail = False

    missing_main =[f for f in c["main_files"] if not _file_ok(main_path, f)]
    if missing_main:
        print(f"🔧 [主体] 缺失 {len(missing_main)} 文件，开始修复...")
        os.makedirs(main_path, exist_ok=True)
        total_needed += len(missing_main)
        for i, f in enumerate(missing_main):
            progress_html.value = f"<span style='color:orange;'>🔧[主体 {i+1}/{len(missing_main)}] {f}</span>"
            print(f"⬇️ {f}...")
            ok, info = download_file(c["main_repo"], f, main_path)
            if ok:
                print(f"   ✅ {info/1024**2:.1f} MB"); total_fixed += 1
            else:
                print(f"   ❌ {info}"); any_fail = True
    else:
        print("✅ 主体文件完整")

    if c.get("te_path") and c.get("te_repo"):
        missing_te =[f for f in c["te_files"] if not _file_ok(c["te_path"], f)]
        if missing_te:
            print(f"\n🔧 [TE] 缺失 {len(missing_te)} 文件，开始修复...")
            os.makedirs(c["te_path"], exist_ok=True)
            total_needed += len(missing_te)
            for i, f in enumerate(missing_te):
                progress_html.value = f"<span style='color:orange;'>🔧[TE {i+1}/{len(missing_te)}] {f}</span>"
                print(f"⬇️ {f}...")
                ok, info = download_file(c["te_repo"], f, c["te_path"])
                if ok:
                    print(f"   ✅ {info/1024**2:.1f} MB"); total_fixed += 1
                else:
                    print(f"   ❌ {info}"); any_fail = True
        else:
            print("✅ TE 文件完整")

    if c.get("transformer_path") and c["transformer_is_gdrive"]:
        tp = c["transformer_path"]
        tw = os.path.join(tp, "diffusion_pytorch_model.safetensors")
        if os.path.exists(tw): print(f"\n✅ GDrive Transformer 存在：{tp}")
        else:
            print(f"\n⚠️  GDrive Transformer 未找到：{tp}")
            print(f"   请确认 Google Drive 已挂载且路径正确（该文件不会自动下载）")
            any_fail = True

    progress_html.value = ""
    print("-" * 40)
    print(f"📊 修复：{total_fixed}/{total_needed}")
    return not any_fail

def _resolve_vae_dir(user_path: str) -> str:
    p = (user_path or "").strip()
    if not p: return ""
    if not os.path.exists(p): raise RuntimeError(f"VAE 路径不存在：{p}")
    if os.path.isdir(p) and os.path.exists(os.path.join(p, "config.json")): return p
    cand = os.path.join(p, "vae")
    if os.path.isdir(cand) and os.path.exists(os.path.join(cand, "config.json")): return cand
    for dirpath, _, fnames in os.walk(p):
        if "config.json" in fnames:
            if "diffusion_pytorch_model.safetensors" in fnames or "diffusion_pytorch_model.bin" in fnames:
                if os.path.basename(dirpath).lower() == "vae": return dirpath
    raise RuntimeError(f"找不到可用 VAE 目录：{p}")

def maybe_override_vae(pipe, vae_path_text, local_only_hint):
    vae_path_text = (vae_path_text or "").strip()
    if not vae_path_text: return False, "VAE：默认（使用模型自带）"
    vae_dir = _resolve_vae_dir(vae_path_text)
    from diffusers import AutoencoderKL
    vae = AutoencoderKL.from_pretrained(
        vae_dir,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        local_files_only=True,
    )
    old = getattr(pipe, "vae", None)
    pipe.vae = vae
    try:
        if torch.cuda.is_available(): pipe.vae.to("cuda")
    except Exception: pass
    try:
        if old is not None: del old
    except Exception: pass
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    gc.collect()
    return True, f"VAE：已覆盖 → {vae_dir}"

def set_pipe_model_root(pipe, main_path, te_path=None, transformer_path=None):
    pipe._model_root = main_path
    pipe._te_path = te_path
    pipe._transformer_override_path = transformer_path

def drop_text_encoder(pipe, drop_tokenizer=False):
    try:
        te = getattr(pipe, "text_encoder", None)
        pipe.text_encoder = None
        if te is not None: del te
    except Exception: pass
    if drop_tokenizer:
        try:
            tok = getattr(pipe, "tokenizer", None)
            pipe.tokenizer = None
            if tok is not None: del tok
        except Exception: pass
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def ensure_text_encoder_loaded(pipe, device="cuda", dtype=torch.float16):
    if getattr(pipe, "text_encoder", None) is not None: return pipe.text_encoder
    model_root = getattr(pipe, "_model_root", None)
    te_path    = getattr(pipe, "_te_path", None)
    if model_root is None: raise RuntimeError("pipe._model_root 未设置")

    is_hybrid = te_path is not None and te_path != model_root
    if is_hybrid:
        te_dir = os.path.join(te_path, "text_encoder")
        if not os.path.exists(te_dir): raise RuntimeError(f"找不到外置 TE 目录：{te_dir}")
        print(f"🔀 混合 TE：使用 {te_dir}")
    else:
        te_dir = os.path.join(model_root, "text_encoder")
        if not os.path.exists(te_dir): raise RuntimeError(f"找不到 text_encoder 目录：{te_dir}")

    from transformers import AutoModel
    from sdnq.loader import apply_sdnq_options_to_model
    import gc

    print("   ⏳ 正在按需加载 Text Encoder (直通显存)...")

    # 提前打扫战场，准备迎接 TE
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    # 🚨 终极修复：保留原版 quantization_config，利用 device_map 直通 GPU！
    # 这样系统内存 (RAM) 几乎不会有波动，权重直接从硬盘流入显存。
    te = AutoModel.from_pretrained(
        te_dir,
        local_files_only=True,
        low_cpu_mem_usage=True,
        device_map=device,          # 直接指向 cuda
        torch_dtype=dtype           # 保持 float16
    )

    te = apply_sdnq_options_to_model(te, use_quantized_matmul=True)

    pipe.text_encoder = te
    print("   ✅ TE 已直通显存，准备编码")
    return te

def ensure_tokenizer_loaded(pipe):
    if getattr(pipe, "tokenizer", None) is not None: return pipe.tokenizer
    root = getattr(pipe, "_model_root", None)
    if root is None: raise RuntimeError("pipe._model_root 未设置")
    tok_dir = os.path.join(root, "tokenizer")
    if not os.path.exists(tok_dir): raise RuntimeError(f"找不到 tokenizer 目录：{tok_dir}")
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(tok_dir, local_files_only=True)
    pipe.tokenizer = tok
    return tok

# ----------------------------
# 🚀 加载模型（核心）
# ----------------------------
def _load_transformer_from_path(transformer_path, transformer_is_gdrive):
    import diffusers, json
    from sdnq.loader import apply_sdnq_options_to_model

    if transformer_is_gdrive and _gdrive_cache_valid():
        print(f"💾 检测到本地缓存，改从本地加载 Transformer：{PATH_BASE_INT8_CACHE}")
        transformer_path = PATH_BASE_INT8_CACHE
        transformer_is_gdrive = False

    gdrive_note = "（Google Drive 直读）" if transformer_is_gdrive else ""
    print(f"🔩 预加载 Transformer：{transformer_path} {gdrive_note}")

    config_path = os.path.join(transformer_path, "config.json")
    if not os.path.exists(config_path): raise RuntimeError(f"找不到 Transformer config.json：{config_path}")

    with open(config_path) as f: trans_cfg = json.load(f)
    class_name = trans_cfg.get("_class_name", "")

    candidate_classes =[]
    if class_name and hasattr(diffusers, class_name): candidate_classes.append(getattr(diffusers, class_name))
    for name in ["ZImageTransformer2DModel", "Transformer2DModel", "DiTTransformer2DModel"]:
        cls = getattr(diffusers, name, None)
        if cls and cls not in candidate_classes: candidate_classes.append(cls)

    last_err = None
    t = None
    for cls in candidate_classes:
        try:
            t = cls.from_pretrained(transformer_path, torch_dtype=torch.float32, local_files_only=True)
            print(f"   ✅ 加载成功：{cls.__name__}")
            break
        except Exception as e:
            last_err = e
            continue

    if t is None:
        try:
            from diffusers import AutoModel as DiffusersAutoModel
            t = DiffusersAutoModel.from_pretrained(transformer_path, torch_dtype=torch.float32, local_files_only=True)
            print(f"   ✅ AutoModel fallback 成功：{type(t).__name__}")
        except Exception as e2:
            raise RuntimeError(f"无法加载 Transformer\n最后错误：{last_err}\nAutoModel 错误：{e2}")

    print(f"   🔧 在 CPU 上执行量化 wrapper（避免 fp32+int8 在 GPU 共存）...")
    t = apply_sdnq_options_to_model(t, use_quantized_matmul=True)
    gc.collect()
    print(f"   ✅ CPU 量化完成：{type(t).__name__}")
    return t

def load_model(main_path, te_path=None, transformer_path=None, transformer_is_gdrive=False):
    global pipe
    import diffusers
    from sdnq.loader import apply_sdnq_options_to_model

    load_kwargs = {"torch_dtype": torch.float32, "device_map": "cuda", "text_encoder": None, "local_files_only": True}

    if transformer_path:
        pre_transformer = _load_transformer_from_path(transformer_path, transformer_is_gdrive)
        load_kwargs["transformer"] = pre_transformer
        print(f"   ☑️  Transformer 已预量化，pipeline 将跳过主体 transformer 目录")

    pipe = diffusers.ZImagePipeline.from_pretrained(main_path, **load_kwargs)
    set_pipe_model_root(pipe, main_path, te_path=te_path, transformer_path=transformer_path)

    if transformer_path:
        try:
            if torch.cuda.is_available(): pipe.transformer.to("cuda")
        except Exception: pass
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        print(f"   ✅ Transformer 就位（已量化）：{type(pipe.transformer).__name__}")
    else:
        print("   ⏳ 正在量化 Pipeline 中的 Transformer...")
        pipe.transformer = apply_sdnq_options_to_model(pipe.transformer, use_quantized_matmul=True)
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    drop_text_encoder(pipe, drop_tokenizer=False)

    try:
        pipe.enable_attention_slicing("max")
        print("   ✅ 注意力切片已启用（max）")
    except Exception as e:
        print(f"   ⚠️ 注意力切片启用失败：{e}")

    assert pipe.transformer is not None, "transformer 加载异常"
    print(f"✅ transformer 就绪：{type(pipe.transformer).__name__}")
    return pipe

# ----------------------------
# 清除与执行操作
# ----------------------------
def hard_cleanup():
    global pipe
    freed =[]

    if "pipe" in globals() and pipe is not None:
        try:
            if getattr(pipe, "text_encoder", None) is not None:
                pipe.text_encoder = None
        except Exception:
            pass
        try:
            if getattr(pipe, "tokenizer", None) is not None:
                pipe.tokenizer = None
        except Exception:
            pass
        try:
            orig = getattr(pipe.scheduler, "__wrapped__", None) or \
                   getattr(getattr(pipe.scheduler, "step", None), "__wrapped__", None)
            if orig is not None:
                pipe.scheduler.step = orig
        except Exception:
            pass
        try:
            del pipe
            freed.append("pipe")
        except Exception:
            pass
        finally:
            pipe = None

    if "loaded_components" in globals():
        try:
            del globals()["loaded_components"]
            freed.append("loaded_components")
        except Exception:
            pass

    if torch.cuda.is_available():
        try: torch.cuda.synchronize()
        except Exception: pass
        try: torch.cuda.empty_cache()
        except Exception: pass
        try: torch.cuda.ipc_collect()
        except Exception: pass
    try:
        gc.collect()
        gc.collect()
    except Exception:
        pass
    return freed

def on_cleanup_click(b):
    output.clear_output()
    cleanup_btn.disabled = True
    with output:
        try:
            status_html.value = "<span style='color:orange;'>🧹 清理中...</span>"
            freed = hard_cleanup()
            print("🧹 清理完成\n释放：", ", ".join(freed) if freed else "（无可释放对象）")
            if torch.cuda.is_available():
                a = torch.cuda.memory_allocated() / 1024**3
                r = torch.cuda.memory_reserved() / 1024**3
                print(f"💾 显存：{a:.2f} GB (使用) / {r:.2f} GB (保留)")
            status_html.value = "<span style='color:green;'>✅ 已清理</span>"
        except Exception as e:
            print("❌ 清理失败:", e)
            status_html.value = "<span style='color:red;'>❌ 清理失败</span>"
        finally:
            cleanup_btn.disabled = False

cleanup_btn.on_click(on_cleanup_click)

def on_execute_click(b):
    global pipe
    output.clear_output()
    execute_btn.disabled = True
    mode = mode_dropdown.value
    c = cfg()
    main_path   = path_input.value.strip()
    vae_path    = vae_path_input.value.strip()
    te_path     = c.get("te_path")
    trans_path  = c.get("transformer_path")
    trans_gdrive= c.get("transformer_is_gdrive", False)

    with output:
        try:
            if mode == "download":
                print(f"⬇️ 下载模式：{model_dropdown.value}\n" + "="*55)
                status_html.value = "<span style='color:orange;'>⏳ 下载中...</span>"
                results, failed = do_download()
                if results["failed"] == 0:
                    print("\n✅ 下载完成！")
                    status_html.value = "<span style='color:green;'>✅ 下载完成</span>"
                else: status_html.value = f"<span style='color:orange;'>⚠️ {results['failed']} 个文件失败</span>"
            elif mode == "repair":
                print(f"🔧 修复模式：{model_dropdown.value}\n" + "="*55)
                status_html.value = "<span style='color:orange;'>⏳ 检查中...</span>"
                ok = do_repair()
                status_html.value = "<span style='color:green;'>✅ 已完整</span>" if ok else "<span style='color:red;'>❌ 仍有文件缺失</span>"
            elif mode == "local":
                print(f"📁 本地加载：{model_dropdown.value}")
                print(f"   主体路径：{main_path}")
                if te_path: print(f"   外置 TE：{te_path}/text_encoder/")
                if trans_path:
                    if trans_gdrive and _gdrive_cache_valid():
                        print(f"   Transformer：{PATH_BASE_INT8_CACHE}（本地缓存，优先使用）")
                    else:
                        print(f"   Transformer：{trans_path} " + ("（GDrive 直读）" if trans_gdrive else ""))
                print("-" * 55)

                missing_main = [f for f in c["main_files"] if not _file_ok(main_path, f)]
                missing_te   =[f for f in c.get("te_files", []) if not _file_ok(te_path or "", f)] if te_path else[]
                if missing_main or missing_te:
                    print("❌ 文件不完整！请切换到下载/修复模式")
                    status_html.value = "<span style='color:red;'>❌ 文件不完整</span>"
                    return
                if trans_path:
                    if not (trans_gdrive and _gdrive_cache_valid()):
                        tw = os.path.join(trans_path, "diffusion_pytorch_model.safetensors")
                        if not os.path.exists(tw):
                            print(f"❌ Transformer 权重不存在：{tw}")
                            status_html.value = "<span style='color:red;'>❌ Transformer 路径无效</span>"
                            return

                status_html.value = "<span style='color:orange;'>⏳ 加载中...</span>"
                if "pipe" in globals() and pipe is not None:
                    print("🧹 检测到旧模型，先执行完整清理...")
                    hard_cleanup()
                    print("🧹 清理完成，开始加载新模型...")

                pipe = load_model(main_path=main_path, te_path=te_path, transformer_path=trans_path, transformer_is_gdrive=trans_gdrive)
                _, vae_msg = maybe_override_vae(pipe, vae_path, local_only_hint=True)
                print("\n✅ 模型加载完成！")
                print(f"📌 方案：{model_dropdown.value}\n🧩 {vae_msg}")
                status_html.value = f"<span style='color:green;'>✅ 已加载</span>"

            if torch.cuda.is_available() and "pipe" in globals() and pipe is not None:
                print(f"\n💾 显存：{torch.cuda.memory_allocated() / 1024**3:.2f} GB / {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

        except Exception as e:
            print(f"\n❌ 错误：{e}")
            status_html.value = "<span style='color:red;'>❌ 失败</span>"
            import traceback; traceback.print_exc()
        finally:
            execute_btn.disabled = False
            progress_html.value = ""
            refresh_local_status()
            refresh_vae_status()
            refresh_cache_status()

execute_btn.on_click(on_execute_click)

# ----------------------------
# UI 布局
# ----------------------------
header = widgets.HTML(value="""
<div style='background:linear-gradient(135deg,#0f2027 0%,#203a43 50%,#2c5364 100%);
            color:white; padding:12px 16px; border-radius:8px; margin-bottom:8px;'>
    <h3 style='margin:0; color:#38bdf8;'>🤖 模型管理器（五合一）</h3>
    <span style='font-size:12px; color:#94a3b8;'>
        uint4 / int8 / 混搭①(uint4-TE+int8) / 混搭②(uint4+GDrive-Transformer) / 混搭③(int8+GDrive-Transformer)
    </span>
</div>
""")

layout = widgets.VBox([
    header, model_dropdown, desc_html, repo_html, local_status_html,
    widgets.HTML("<hr style='margin:8px 0; border-color:#334155;'>"),
    mode_dropdown, path_input, vae_path_input, vae_status_html,
    widgets.HBox([execute_btn, cleanup_btn, cache_btn, status_html],
                 layout=widgets.Layout(gap="10px", align_items="center")),
    cache_status_html,
    progress_html, output
], layout=widgets.Layout(padding="12px", border="1px solid #334155", border_radius="10px", width="760px"))

display(layout)

In [ ]:
# @title 🧩 Lora-Distill 下载/转换/融合/卸载
import ipywidgets as widgets
from IPython.display import display
import torch, os, gc, re, requests
from pathlib import Path
import safetensors.torch
from huggingface_hub import hf_hub_download, list_repo_files

# ════════════════════════════════════════════════════════
# 1. 转换映射
# ════════════════════════════════════════════════════════
ATTENTION_MAP = {
    "attention_to_q":     "attention.to_q",
    "attention_to_k":     "attention.to_k",
    "attention_to_v":     "attention.to_v",
    "attention_to_out_0": "attention.to_out.0",
    "attention_to_out":   "attention.to_out.0",
}
FF_MAP = {
    "feed_forward_w1": "feed_forward.w1",
    "feed_forward_w2": "feed_forward.w2",
    "feed_forward_w3": "feed_forward.w3",
}
SUB_MAP    = {**ATTENTION_MAP, **FF_MAP}
SUFFIX_MAP = {
    "lora_down.weight": "lora_A.weight",
    "lora_up.weight":   "lora_B.weight",
}
PATTERN = re.compile(
    r"^lora_unet__(layers|context_refiner|noise_refiner)"
    r"_(\d+)_(attention_to_\w+|feed_forward_w\d)"
    r"\.(lora_down\.weight|lora_up\.weight|alpha)$"
)

def convert_key(old_key):
    m = PATTERN.match(old_key)
    if not m: return None, "skip"
    block_type, idx, sub, suffix = m.groups()
    if suffix == "alpha": return None, "alpha"
    mapped_sub    = SUB_MAP.get(sub)
    mapped_suffix = SUFFIX_MAP.get(suffix)
    if not mapped_sub or not mapped_suffix: return None, "skip"
    return f"diffusion_model.{block_type}.{idx}.{mapped_sub}.{mapped_suffix}", "ok"

def needs_conversion(file_path):
    try:
        tensors = safetensors.torch.load_file(file_path, device="cpu")
        return any(PATTERN.match(k) for k in tensors.keys())
    except Exception:
        return False

# ════════════════════════════════════════════════════════
# 2. 下载工具
# ════════════════════════════════════════════════════════
LORA_DIR = "/content/Lora"

def download_from_url(url, custom_filename=None, force=False):
    os.makedirs(LORA_DIR, exist_ok=True)
    filename = custom_filename or os.path.basename(url.split("?")[0])
    if not filename.endswith(".safetensors"): filename += ".safetensors"
    local_path = os.path.join(LORA_DIR, filename)
    if os.path.exists(local_path) and not force: return local_path, "already_exists"
    try:
        r = requests.get(url, stream=True, timeout=30)
        r.raise_for_status()
        with open(local_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192): f.write(chunk)
        return local_path, "downloaded"
    except Exception as e:
        return None, str(e)

def download_from_hub(repo_id, filename, force=False):
    os.makedirs(LORA_DIR, exist_ok=True)
    local_path = os.path.join(LORA_DIR, os.path.basename(filename))
    if os.path.exists(local_path) and not force: return local_path, "already_exists"
    try:
        downloaded = hf_hub_download(
            repo_id=repo_id, filename=filename,
            local_dir=LORA_DIR, local_dir_use_symlinks=False, force_download=force,
        )
        return downloaded, "downloaded"
    except Exception as e:
        return None, str(e)

def list_repo_safetensors(repo_id):
    try: return[f for f in list_repo_files(repo_id) if f.endswith(".safetensors")]
    except Exception: return[]

# ════════════════════════════════════════════════════════
# 3. 转换函数
# ════════════════════════════════════════════════════════
def convert_lora_file(src_path, delete_original=False, log=print):
    dst_path = str(Path(src_path).parent / (Path(src_path).stem + "_diffusers.safetensors"))
    if os.path.exists(dst_path):
        os.remove(dst_path)
        log(f"🗑️ 已删除旧转换文件: {Path(dst_path).name}")
    try:
        sd = safetensors.torch.load_file(src_path, device="cpu")
        converted, dropped, skipped = {}, [],[]
        for k, v in sd.items():
            new_key, status = convert_key(k)
            if status == "ok": converted[new_key] = v
            elif status == "alpha": dropped.append(k)
            else: skipped.append(k)
        log(f"✅ 转换: {len(converted)}  丢弃(alpha): {len(dropped)}  跳过: {len(skipped)}")
        safetensors.torch.save_file(converted, dst_path)
        size_mb = Path(dst_path).stat().st_size / 1024 / 1024
        log(f"✅ 已保存: {Path(dst_path).name}  ({size_mb:.1f} MB)")
        if delete_original:
            os.remove(src_path)
            log(f"🗑️ 已删除原文件: {Path(src_path).name}")
        return dst_path
    except Exception as e:
        log(f"❌ 转换失败: {e}")
        return None

# ════════════════════════════════════════════════════════
# 4. 融合 / 卸载 (防残留版)
# ════════════════════════════════════════════════════════
if "_fused_state" not in globals():
    _fused_state = {}

def _make_lora_hook(A, B, scale, is_ff):
    def hook(module, input, output):
        x = input[0].to(dtype=torch.float32)
        lora_out = (x @ A.T.to(x.device)) @ B.T.to(x.device)
        lora_out = lora_out * scale
        return output + lora_out.to(dtype=output.dtype)
    return hook

def unfuse_lora(log=print):
    global _fused_state

    # A. 正常卸载
    if _fused_state.get("hooks"):
        for h in _fused_state["hooks"]: h.remove()
        log(f"✅ 正常卸载: 已移除 {len(_fused_state['hooks'])} 个 LoRA hook")

    # B. 底层暴力兜底扫荡
    if "pipe" in globals():
        target_model = getattr(pipe, "transformer", getattr(pipe, "unet", None))
        cleared_count = 0
        if target_model is not None:
            for name, module in target_model.named_modules():
                if hasattr(module, "_forward_hooks") and len(module._forward_hooks) > 0:
                    cleared_count += len(module._forward_hooks)
                    module._forward_hooks.clear()

        if cleared_count > 0:
            log(f"🧹 深度清理: 铲除了 {cleared_count} 个残留的幽灵 Hook！")

    _fused_state = {}
    gc.collect()

def fuse_lora(lora_path, scale=1.0, log=print):
    global _fused_state
    log("🔄 融合前环境自检与清理...")
    unfuse_lora(log=log)

    sd = safetensors.torch.load_file(lora_path, device="cpu")
    module_map = dict(pipe.transformer.named_modules())

    base_keys = set()
    for k in sd:
        for sfx in[".lora_A.weight", ".lora_B.weight"]:
            if k.endswith(sfx):
                base_keys.add(k[:-len(sfx)])
                break

    hooks =[]
    n_attn, n_ff, n_fail = 0, 0, 0
    for bk in base_keys:
        bare = bk.removeprefix("diffusion_model.").removeprefix("transformer.")
        module = module_map.get(bare)
        if module is None:
            n_fail += 1
            continue

        A = sd[bk + ".lora_A.weight"].float()
        B = sd[bk + ".lora_B.weight"].float()
        is_ff = "feed_forward" in bk

        hook_fn = _make_lora_hook(A, B, scale, is_ff)
        h = module.register_forward_hook(hook_fn)
        hooks.append(h)

        if is_ff: n_ff += 1
        else: n_attn += 1

    total = n_attn + n_ff
    log(f"✅ 已注册 LoRA hook {total} 个  (attention={n_attn}, feed_forward={n_ff}, 失败={n_fail})")
    _fused_state = {"file": Path(lora_path).name, "scale": scale, "sd": sd, "hooks": hooks}

# ════════════════════════════════════════════════════════
# 5. UI 构建 (新增本地文件选择)
# ════════════════════════════════════════════════════════
output_widget = widgets.Output()

# ── 来源选择 ──────────────────────────────────────────
source_method = widgets.RadioButtons(
    options=[("📂 本地已有文件", "local"), ("📦 Repo ID 下载", "repo"), ("🔗 URL 下载", "url")],
    value="local",
    description="文件来源:",
    layout=widgets.Layout(width="400px")
)

# 【新增】本地文件下拉选单
local_file_dropdown = widgets.Dropdown(options=[], layout=widgets.Layout(flex="1 1 auto"))
local_file_row = widgets.HBox([widgets.Label("选中文件:", layout=widgets.Layout(width="70px")), local_file_dropdown],
    layout=widgets.Layout(width="100%") # 默认选中 local，所以默认显示
)

# Repo / URL 输入
url_input = widgets.Text(placeholder="https://.../model.safetensors", layout=widgets.Layout(flex="1 1 auto"))
url_row = widgets.HBox([widgets.Label("URL:", layout=widgets.Layout(width="70px")), url_input], layout=widgets.Layout(width="100%", display="none"))

repo_input = widgets.Text(value="alibaba-pai/Z-Image-Fun-Lora-Distill", layout=widgets.Layout(flex="1 1 auto"))
browse_btn = widgets.Button(description="📂 浏览文件", layout=widgets.Layout(width="110px"))
repo_row = widgets.HBox([widgets.Label("Repo ID:", layout=widgets.Layout(width="70px")), repo_input, browse_btn], layout=widgets.Layout(width="100%", display="none"))

filename_input = widgets.Text(placeholder="文件名（例如: lora.safetensors）", layout=widgets.Layout(flex="1 1 auto"))
filename_row = widgets.HBox([widgets.Label("文件名:", layout=widgets.Layout(width="70px")), filename_input], layout=widgets.Layout(width="100%", display="none"))

file_selector = widgets.Select(options=[], rows=6, layout=widgets.Layout(flex="1 1 auto", display="none"))
confirm_file_btn = widgets.Button(description="✅ 确认", button_style="success", layout=widgets.Layout(width="80px", display="none"))
file_select_row = widgets.HBox([widgets.Label("选择:", layout=widgets.Layout(width="70px")), file_selector, confirm_file_btn], layout=widgets.Layout(width="100%"))

# ── 选项 ──────────────────────────────────────────────
force_check   = widgets.Checkbox(value=False, description="强制重新下载", indent=False)
del_orig_check = widgets.Checkbox(value=True,  description="转换后删除原文件", indent=False)
opts_row = widgets.HBox([force_check, del_orig_check], layout=widgets.Layout(display="none")) # 选本地时隐藏

# ── Merge Scale ───────────────────────────────────────
scale_slider = widgets.FloatSlider(
    value=1.0, min=0.0, max=2.0, step=0.05,
    description="Merge Scale:",
    readout_format=".2f",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px")
)

# ── 操作模式 ──────────────────────────────────────────
mode_dropdown = widgets.Dropdown(
    options=[
        ("🔗  融合到模型 (最常用)",  "fuse"),
        ("🔧  仅转换",               "convert"),
        ("🔧🔗 转换 + 融合",         "convert_fuse"),
        ("⬇️🔧🔗 下载+转换+融合",   "download_convert_fuse"),
        ("⬇️  仅下载",               "download"),
        ("🔓  强力卸载当前 LoRA",   "unfuse"),
    ],
    value="fuse",
    description="执行操作:",
    layout=widgets.Layout(width="320px")
)

execute_btn = widgets.Button(description="🚀 执行", button_style="primary", layout=widgets.Layout(width="110px", height="38px"))
refresh_btn = widgets.Button(description="🔄 刷新列表", layout=widgets.Layout(width="110px", height="38px"))
status_html    = widgets.HTML(value="<span style='color:gray;'>准备就绪</span>")
fused_html     = widgets.HTML(value="<span style='color:gray;'>未融合</span>")
file_list_html = widgets.HTML(value="<i>暂无文件</i>")

# ── 动态 UI 事件 ──────────────────────────────────────
def on_method_change(change):
    method = change["new"]
    # 控制行显示隐藏
    local_file_row.layout.display = "flex" if method == "local" else "none"
    url_row.layout.display        = "flex" if method == "url" else "none"
    repo_row.layout.display       = "flex" if method == "repo" else "none"
    filename_row.layout.display   = "flex" if method != "local" else "none"
    opts_row.layout.display       = "flex" if method != "local" else "none"
    file_selector.layout.display    = "none"
    confirm_file_btn.layout.display = "none"

    # 智能切换默认操作模式
    if method == "local":
        mode_dropdown.value = "fuse"
    else:
        mode_dropdown.value = "download_convert_fuse"

source_method.observe(on_method_change, names="value")

def on_browse(b):
    repo = repo_input.value.strip()
    if not repo:
        with output_widget: output_widget.clear_output(); print("❌ 请填写 Repo ID")
        return
    with output_widget:
        output_widget.clear_output(); print(f"📡 获取 {repo} 文件列表...")
        files = list_repo_safetensors(repo)
        if not files: print("❌ 未找到文件"); file_selector.options =[]
        else:
            file_selector.options = files
            file_selector.value   = files[0]
            file_selector.layout.display    = "flex"
            confirm_file_btn.layout.display = "flex"
            print(f"✅ 找到 {len(files)} 个文件，请选择")

browse_btn.on_click(on_browse)

def on_confirm_file(b):
    if file_selector.value:
        filename_input.value = file_selector.value
        file_selector.layout.display = "none"
        confirm_file_btn.layout.display = "none"
        with output_widget: output_widget.clear_output(); print(f"✅ 已选择: {file_selector.value}")

confirm_file_btn.on_click(on_confirm_file)

def refresh_file_list():
    if not os.path.exists(LORA_DIR):
        file_list_html.value = "<i>目录不存在</i>"
        local_file_dropdown.options =[]
        return

    files = [f for f in os.listdir(LORA_DIR) if f.endswith(".safetensors")]

    # 更新下拉列表
    current_val = local_file_dropdown.value
    local_file_dropdown.options = sorted(files)
    if current_val in files: local_file_dropdown.value = current_val
    elif files: local_file_dropdown.value = sorted(files)[0]

    # 更新状态显示
    if not files:
        file_list_html.value = "<i>Lora 文件夹为空</i>"
        return

    lines =[]
    for f in sorted(files):
        path = os.path.join(LORA_DIR, f)
        size = os.path.getsize(path) / 1024 / 1024
        mark = "⚠️ 需转换" if needs_conversion(path) else "✅ 可用"
        lines.append(f"<div>{mark}  {f}  ({size:.2f} MB)</div>")

    if _fused_state.get("file"):
        fused_html.value = f"<span style='color:green;'>🔗 当前挂载中: {_fused_state['file']}  (scale={_fused_state['scale']})</span>"
    else:
        fused_html.value = "<span style='color:gray;'>未挂载任何 LoRA</span>"

    file_list_html.value = "<b>Lora 本地文件列表:</b><br>" + "".join(lines)

refresh_btn.on_click(lambda b: refresh_file_list())

# ── 核心执行逻辑 ──────────────────────────────────────
def on_execute(b):
    output_widget.clear_output()
    execute_btn.disabled = True
    mode   = mode_dropdown.value
    method = source_method.value
    force  = force_check.value
    del_orig = del_orig_check.value
    scale  = scale_slider.value

    def log(msg):
        with output_widget: print(msg)

    try:
        status_html.value = "<span style='color:orange;'>⏳ 执行中...</span>"

        # ── 0. 卸载优先 ──
        if mode == "unfuse":
            unfuse_lora(log=log)
            status_html.value = "<span style='color:green;'>✅ 卸载/清理完成</span>"
            refresh_file_list()
            return

        target_path = None

        # ── 1. 获取目标文件 ──
        if method == "local":
            if not local_file_dropdown.value:
                log("❌ 本地无可用文件，请先切换来源进行下载"); status_html.value = "❌"; return
            target_path = os.path.join(LORA_DIR, local_file_dropdown.value)
            log(f"📂 选用文件: {local_file_dropdown.value}")
        else:
            if "download" not in mode:
                log("⚠️ 提示: 你选择了网络来源，但操作未包含下载。将自动尝试下载...")
            if method == "url":
                url = url_input.value.strip()
                if not url: log("❌ 请输入 URL"); status_html.value = "❌"; return
                log(f"⬇️ URL下载中...")
                local_path, result = download_from_url(url, filename_input.value.strip() or None, force)
            else:
                repo, fname = repo_input.value.strip(), filename_input.value.strip()
                if not repo or not fname: log("❌ 请填写 Repo ID 和文件名"); status_html.value = "❌"; return
                log(f"⬇️ Hub下载中: {repo}/{fname}")
                local_path, result = download_from_hub(repo, fname, force)

            if local_path:
                log(f"{'✅ 下载成功' if result == 'downloaded' else '⏭️ 文件已存在'}: {local_path}")
                target_path = local_path
            else:
                log(f"❌ 下载失败: {result}"); status_html.value = "❌"; return

        # ── 2. 转换 ──
        converted_path = None
        if "convert" in mode:
            if needs_conversion(target_path):
                log(f"\n🔧 开始转换: {Path(target_path).name}")
                converted_path = convert_lora_file(target_path, del_orig, log)
            else:
                log("⏭️ 该文件无需转换 (已是兼容格式)")
                converted_path = target_path

        # ── 3. 融合 ──
        if "fuse" in mode:
            fuse_src = converted_path or target_path
            if needs_conversion(fuse_src):
                log("⚠️ 警告：要融合的文件似乎未经转换，这可能会导致报错！")
            log(f"\n🔗 开始融合: {Path(fuse_src).name}  (scale={scale:.2f})")
            fuse_lora(fuse_src, scale=scale, log=log)

        status_html.value = "<span style='color:green;'>✅ 任务圆满完成</span>"
        refresh_file_list()

    except Exception as e:
        log(f"\n❌ 错误: {e}")
        import traceback; traceback.print_exc()
        status_html.value = "<span style='color:red;'>❌ 运行失败</span>"
    finally:
        execute_btn.disabled = False

execute_btn.on_click(on_execute)
refresh_file_list()

# ════════════════════════════════════════════════════════
# 6. 最终界面布局
# ════════════════════════════════════════════════════════
header = widgets.HTML("""
<div style='background:linear-gradient(135deg,#11998e,#38ef7d); color:white;padding:12px 16px;border-radius:8px;margin-bottom:8px;'>
  <h3 style='margin:0;'>🧩 LoRA 控制台 (增强版)</h3>
  <span style='font-size:12px;'>直接下拉选择本地文件加载 | 完美解决幽灵残留 | 下载转换一站式</span>
</div>
""")

ui = widgets.VBox([
    header,
    source_method,
    local_file_row,
    url_row,
    repo_row,
    filename_row,
    file_select_row,
    opts_row,
    scale_slider,
    mode_dropdown,
    widgets.HBox([execute_btn, refresh_btn]),
    status_html,
    fused_html,
    file_list_html,
    output_widget,
], layout=widgets.Layout(padding="12px", border="1px solid #ddd", border_radius="10px", width="100%", max_width="900px"))

display(ui)

In [ ]:
# @title 🖌️🖌️ Z-Image 出图面板
import os, gc, time, json, numpy as np, torch, traceback, inspect
from datetime import datetime
from PIL import Image
from IPython.display import display, clear_output
import ipywidgets as widgets
from google.colab import drive

# ========= 依赖：模型管理器的 3 个函数 =========
def _require_manager_funcs():
    needed =["ensure_tokenizer_loaded", "ensure_text_encoder_loaded", "drop_text_encoder"]
    missing =[n for n in needed if n not in globals()]
    if missing:
        raise RuntimeError("缺少模型管理器函数（先运行模型管理器并加载模型）: " + ", ".join(missing))
    return (globals()["ensure_tokenizer_loaded"],
            globals()["ensure_text_encoder_loaded"],
            globals()["drop_text_encoder"])

def _get_existing_pipe():
    if "pipe" in globals() and globals().get("pipe", None) is not None:
        return globals()["pipe"]
    if "loaded_components" in globals() and isinstance(globals()["loaded_components"], dict):
        return globals()["loaded_components"].get("pipe", None)
    return None

def _round_to_16(x: int) -> int:
    return int(round(x / 16) * 16)

def _pipe_accepts_kw(p, key: str) -> bool:
    try:
        sig = inspect.signature(p.__call__)
        return key in sig.parameters
    except Exception:
        return False

# ========= allocator/显存小工具 =========
def _get_allocator_conf():
    return os.environ.get("PYTORCH_CUDA_ALLOC_CONF", "")

def _allocator_status_text():
    conf = _get_allocator_conf()
    if not conf:
        return ("<span style='color:orange;'>⚠️ allocator: 未设置 PYTORCH_CUDA_ALLOC_CONF</span>"
                "<br><span style='color:#64748b;font-size:12px;'>建议重启 runtime 后最先设置："
                "expandable_segments:True,max_split_size_mb:128</span>")
    ok = ("expandable_segments:True" in conf)
    if ok:
        return (f"<span style='color:green;'>✅ allocator: {conf}</span>"
                "<br><span style='color:#64748b;font-size:12px;'>已启用 expandable_segments（更抗碎片，适合冲大图）</span>")
    return (f"<span style='color:orange;'>⚠️ allocator: {conf}</span>"
            "<br><span style='color:#64748b;font-size:12px;'>建议包含 expandable_segments:True（需要重启后生效）</span>")

def _print_cuda_mem_line(tag=""):
    if not torch.cuda.is_available():
        return
    alloc = torch.cuda.memory_allocated() / (1024**3)
    resv  = torch.cuda.memory_reserved() / (1024**3)
    peak  = torch.cuda.max_memory_allocated() / (1024**3)
    if tag:
        print(f"[{tag}] ", end="")
    print(f"allocated={alloc:.2f} GB | reserved={resv:.2f} GB | peak={peak:.2f} GB")

def runtime_fragmentation_cleanup(synchronize=True, reset_peak=False):
    if not torch.cuda.is_available():
        return
    if synchronize:
        try: torch.cuda.synchronize()
        except Exception: pass
    try: gc.collect()
    except Exception: pass
    try: torch.cuda.empty_cache()
    except Exception: pass
    try: torch.cuda.ipc_collect()
    except Exception: pass
    if reset_peak:
        try: torch.cuda.reset_peak_memory_stats()
        except Exception: pass
    if synchronize:
        try: torch.cuda.synchronize()
        except Exception: pass

# ========= SDPA 上下文 =========
def _enter_sdpa_efficient():
    try:
        from torch.nn.attention import sdpa_kernel, SDPBackend
        ctx = sdpa_kernel([SDPBackend.EFFICIENT_ATTENTION, SDPBackend.MATH])
        ctx.__enter__()
        return ctx
    except Exception:
        return None

def _exit_sdpa(ctx):
    if ctx is None: return
    try: ctx.__exit__(None, None, None)
    except Exception: pass

# ========= dtype / 小工具 =========
def _dtype_from_mode(mode: str):
    mode = (mode or "fp32").lower()
    if mode == "fp16": return torch.float16
    if mode == "bf16": return torch.bfloat16
    return torch.float32

def _bf16_is_supported_cuda():
    if not torch.cuda.is_available(): return False
    try: return bool(torch.cuda.is_bf16_supported())
    except Exception: return False

def _maybe_cast_pipe_modules(p, target_dtype: torch.dtype, enable: bool):
    if not enable: return
    if not torch.cuda.is_available(): return
    try:
        if hasattr(p, "transformer") and p.transformer is not None:
            p.transformer.to("cuda", dtype=target_dtype)
    except Exception: pass
    try:
        if hasattr(p, "vae") and p.vae is not None:
            p.vae.to("cuda", dtype=target_dtype)
    except Exception: pass

def _prepare_full_gpu(p, attn_slice):
    if hasattr(p, "reset_device_map"):
        try: p.reset_device_map()
        except Exception: pass
    if not torch.cuda.is_available():
        raise RuntimeError("没有检测到 CUDA")
    runtime_fragmentation_cleanup(synchronize=True, reset_peak=False)
    p.to("cuda")
    if hasattr(p, "enable_attention_slicing"):
        if attn_slice == "auto":
            attn_slice = "max"
        p.enable_attention_slicing(attn_slice)

def _apply_int8_matmul_to_transformer(p, enable: bool):
    try:
        from sdnq.loader import apply_sdnq_options_to_model
    except Exception as e:
        return False, f"未能导入 sdnq.loader: {e}"
    if not hasattr(p, "transformer") or p.transformer is None:
        return False, "pipe.transformer 不存在"
    try:
        p.transformer = apply_sdnq_options_to_model(p.transformer, use_quantized_matmul=bool(enable))
        return True, f"INT8 MatMul = {bool(enable)}"
    except Exception as e:
        return False, f"设置失败: {e}"

def _encode_prompt_embeds_then_drop_te(p, prompt_text: str, negative_text: str | None, cfg: float, te_dtype=torch.float16):
    ensure_tokenizer_loaded_fn, ensure_text_encoder_loaded_fn, drop_text_encoder_fn = _require_manager_funcs()

    if not _pipe_accepts_kw(p, "prompt_embeds"):
        raise RuntimeError("pipe 不支持 prompt_embeds")

    ensure_tokenizer_loaded_fn(p)
    te = ensure_text_encoder_loaded_fn(p, device="cuda", dtype=te_dtype)

    encode_fn = getattr(p, "encode_prompt", None) or getattr(p, "_encode_prompt", None)
    if encode_fn is None:
        raise RuntimeError("pipe 没有 encode_prompt/_encode_prompt")

    sig = inspect.signature(encode_fn)
    keys = set(sig.parameters.keys())

    def _encode_one(text: str):
        kwargs = {}
        if "prompt" in keys: kwargs["prompt"] = text
        if "device" in keys: kwargs["device"] = torch.device("cuda")
        if "num_images_per_prompt" in keys: kwargs["num_images_per_prompt"] = 1
        if "do_classifier_free_guidance" in keys: kwargs["do_classifier_free_guidance"] = False
        if "negative_prompt" in keys: kwargs["negative_prompt"] = None

        encoded = encode_fn(**kwargs)
        out = {}
        if isinstance(encoded, torch.Tensor):
            out["prompt_embeds"] = encoded
        elif isinstance(encoded, (tuple, list)):
            out["prompt_embeds"] = encoded[0]
            if len(encoded) >= 3 and isinstance(encoded[2], torch.Tensor):
                out["pooled_prompt_embeds"] = encoded[2]
        elif isinstance(encoded, dict):
            for k in["prompt_embeds", "pooled_prompt_embeds", "attention_mask"]:
                if k in encoded: out[k] = encoded[k]
        return out

    do_cfg = bool(cfg is not None and float(cfg) >= 1.0)
    pos = _encode_one(prompt_text)
    embeds_kwargs = {"prompt_embeds": pos["prompt_embeds"]}
    if "pooled_prompt_embeds" in pos:
        embeds_kwargs["pooled_prompt_embeds"] = pos["pooled_prompt_embeds"]

    if do_cfg:
        neg_text = negative_text if (negative_text and negative_text.strip()) else ""
        neg = _encode_one(neg_text)
        embeds_kwargs["negative_prompt_embeds"] = neg["prompt_embeds"]
        if "pooled_prompt_embeds" in neg:
            embeds_kwargs["negative_pooled_prompt_embeds"] = neg["pooled_prompt_embeds"]

    if 'te' in locals():
        del te

    drop_text_encoder_fn(p, drop_tokenizer=False)

    for k, v in list(embeds_kwargs.items()):
        if isinstance(v, torch.Tensor):
            embeds_kwargs[k] = v.to("cuda")
    return embeds_kwargs

# 💡 核心强化：支持极限低显存模式的 VAE 解码
def _decode_latent_with_pipe_vae(p, latent_t: torch.Tensor, scaling_factor: float, tiled: bool, tile_size, extreme_vram_mode: bool = False):
    if not hasattr(p, "vae") or p.vae is None:
        raise RuntimeError("pipe.vae 不存在")
    vae = p.vae

    if extreme_vram_mode:
        print("   🐌 [极限模式] 将 VAE 移至 CPU 解码 (时间较长，但绝对防 OOM，请耐心等待)...")
        vae.to("cpu", dtype=torch.float32)
        lat = latent_t.clone().to("cpu", dtype=torch.float32)
        if lat.dim() == 3: lat = lat.unsqueeze(0)
        with torch.no_grad():
            if tiled and hasattr(vae, "enable_tiling"):
                vae.enable_tiling()
                try: vae.tile_sample_size = int(tile_size)
                except Exception: pass
            decoded = vae.decode(lat / float(scaling_factor)).sample
    else:
        vae.to("cuda", dtype=torch.float16)
        lat = latent_t
        if lat.dim() == 3: lat = lat.unsqueeze(0)
        lat = lat.to("cuda", dtype=torch.float16)
        with torch.no_grad():
            if tiled and hasattr(vae, "enable_tiling"):
                vae.enable_tiling()
                try: vae.tile_sample_size = int(tile_size)
                except Exception: pass
            decoded = vae.decode(lat / float(scaling_factor)).sample

    img_np = (decoded[0] / 2 + 0.5).clamp(0, 1).detach().cpu().permute(1, 2, 0).float().numpy()
    del decoded, lat
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return Image.fromarray((img_np * 255).astype(np.uint8))

# ========= 图像比例预设 =========
ASPECT_RATIOS = {
    "自定义": None,
    "1:1 方形 (1024×1024)": (1024, 1024),
    "4:3 横屏 (1152×864)": (1152, 864),
    "3:4 竖屏 (864×1152)": (864, 1152),
    "16:9 宽屏 (1344×768)": (1344, 768),
    "9:16 手机屏 (768×1344)": (768, 1344),
    "3:2 照片横 (1216×816)": (1216, 816),
    "2:3 照片竖 (816×1216)": (816, 1216),
    "21:9 超宽 (1536×656)": (1536, 656),
}

def _auto_tile_for_size(w, h):
    m = max(w, h)
    if m >= 1536: return 512
    if m >= 1024: return 768
    return 1024

if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

STYLE_CSS = """
<style>
.zimg-panel { font-family: 'Segoe UI', sans-serif; }
.zimg-section {
    background: #1e293b; border-radius: 12px; padding: 16px; margin: 8px 0;
    border: 1px solid #334155;
}
.zimg-section-title {
    color: #38bdf8; font-size: 14px; font-weight: 600;
    margin-bottom: 12px; padding-bottom: 8px;
    border-bottom: 1px solid #334155;
}
.zimg-header {
    background: linear-gradient(135deg, #0ea5e9 0%, #8b5cf6 50%, #ec4899 100%);
    border-radius: 16px; padding: 20px; margin-bottom: 16px;
    box-shadow: 0 4px 20px rgba(14, 165, 233, 0.3);
}
.zimg-header h1 { color: white; font-size: 24px; font-weight: 700; margin: 0 0 4px 0; }
.zimg-header p { color: rgba(255,255,255,0.85); font-size: 12px; margin: 0; }
.zimg-label { color: #94a3b8; font-size: 12px; margin-bottom: 4px; }
.zimg-info { color: #64748b; font-size: 11px; margin-top: 4px; }
</style>
"""

header_html = widgets.HTML(STYLE_CSS + """
<div class="zimg-panel">
<div class="zimg-header">
    <h1>🎨 Z-Image Turbo</h1>
    <p>直通显存TE · 极限大图防护 · VAE 断崖式优化</p>
</div>
</div>
""")

allocator_html = widgets.HTML(value=f"""
<div style="margin:10px 0 0 0; padding:10px 12px; border:1px solid #334155; border-radius:12px; background:#0b1220;">
  <div style="font-size:13px; color:#e2e8f0; font-weight:600; margin-bottom:4px;">🧠 Boot/Allocator 状态</div>
  <div style="font-size:13px;">{_allocator_status_text()}</div>
</div>
""")

prompt_box = widgets.Textarea(value="a beautiful cat sitting on a windowsill, soft sunlight, detailed fur, photorealistic", placeholder="输入正向提示词...", layout=widgets.Layout(width="100%", height="300px"))
neg_box = widgets.Textarea(value="", placeholder="输入负面提示词（CFG≤1无效）...", layout=widgets.Layout(width="100%", height="60px"))

w_png_meta_path = widgets.Text(value="", placeholder="填写要读取元数据的 PNG 路径（例如 /content/drive/MyDrive/Z-image/Outputs/img_xxx.png）", description="PNG 路径", style={'description_width': '90px'}, layout=widgets.Layout(width="100%"))
btn_load_meta = widgets.Button(description="📥 一键读取 PNG 元数据并回填参数", button_style="info", layout=widgets.Layout(width="320px", height="36px"))
btn_copy_meta_json = widgets.Button(description="📋 复制元数据 JSON 到剪贴板", button_style="", layout=widgets.Layout(width="240px", height="36px"))
meta_status = widgets.HTML(value="<div class='zimg-info'>在此填 PNG 路径后，可一键读取 parameters 元数据并回填面板参数。</div>")

def _read_metadata_from_png(png_path: str) -> dict | None:
    try:
        img = Image.open(png_path)
        if hasattr(img, "info") and "parameters" in img.info:
            return json.loads(img.info["parameters"])
    except Exception as e:
        print(f"读取 PNG 元数据失败: {e}")
    return None

def _safe_float(v, default=None):
    try:
        if v is None: return default
        return float(v)
    except Exception: return default

def _safe_int(v, default=None):
    try:
        if v is None: return default
        return int(float(v))
    except Exception: return default

def _set_ratio_preset_if_exact_match(width: int, height: int) -> str:
    for name, wh in ASPECT_RATIOS.items():
        if wh is None: continue
        if int(wh[0]) == int(width) and int(wh[1]) == int(height): return name
    return "自定义"

def _apply_metadata_to_ui(meta: dict):
    if isinstance(meta.get("prompt", None), str): prompt_box.value = meta["prompt"]
    if isinstance(meta.get("negative_prompt", None), str): neg_box.value = meta["negative_prompt"]
    steps = _safe_int(meta.get("steps", None), default=None)
    if steps is not None: w_steps.value = int(max(w_steps.min, min(w_steps.max, steps)))
    cfg = _safe_float(meta.get("cfg_scale", None), default=None)
    if cfg is not None: w_cfg.value = float(max(w_cfg.min, min(w_cfg.max, cfg)))
    seed = _safe_int(meta.get("seed", None), default=None)
    if seed is not None: w_seed.value = int(seed)
    width = _safe_int(meta.get("width", None), default=None)
    height = _safe_int(meta.get("height", None), default=None)
    if width is not None and height is not None:
        width16 = _round_to_16(int(width))
        height16 = _round_to_16(int(height))
        width16 = int(max(w_width.min, min(w_width.max, width16)))
        height16 = int(max(w_height.min, min(w_height.max, height16)))
        ratio_name = meta.get("aspect_ratio_preset", None)
        if not isinstance(ratio_name, str) or ratio_name not in ASPECT_RATIOS:
            ratio_name = _set_ratio_preset_if_exact_match(width16, height16)
        w_ratio.value = ratio_name
        w_width.value = width16
        w_height.value = height16
        _update_size_info(None)

    svh = meta.get("seed_variance_enhancer", {})
    if isinstance(svh, dict):
        w_svh_enable.value = bool(svh.get("enabled", False))
        rp = _safe_float(svh.get("randomize_percent", None), default=None)
        if rp is not None: w_svh_randomize_percent.value = float(max(w_svh_randomize_percent.min, min(w_svh_randomize_percent.max, rp)))
        st = _safe_float(svh.get("strength", None), default=None)
        if st is not None: w_svh_strength.value = float(st)
        ni = svh.get("noise_insert", None)
        if isinstance(ni, str) and ni in[o for o in w_svh_noise_insert.options]: w_svh_noise_insert.value = ni
        sw = _safe_float(svh.get("steps_switchover_percent", None), default=None)
        if sw is not None: w_svh_steps_switchover_percent.value = float(max(w_svh_steps_switchover_percent.min, min(w_svh_steps_switchover_percent.max, sw)))
        ms = svh.get("mask_starts_at", None)
        if isinstance(ms, str) and ms in[o for o in w_svh_mask_starts_at.options]: w_svh_mask_starts_at.value = ms
        mp = _safe_float(svh.get("mask_percent", None), default=None)
        if mp is not None: w_svh_mask_percent.value = float(max(w_svh_mask_percent.min, min(w_svh_mask_percent.max, mp)))
        sm = svh.get("seed_mode", None)
        if isinstance(sm, str) and sm in[v for _, v in w_svh_seed_mode.options]: w_svh_seed_mode.value = sm
        sseed = _safe_int(svh.get("seed", None), default=None)
        if sseed is not None: w_svh_seed.value = int(sseed)

    lora = meta.get("lora_slots", {})
    if isinstance(lora, dict):
        def _set_if_in_options(dd: widgets.Dropdown, val: str):
            vals = [v for _, v in dd.options]
            dd.value = val if (val in vals) else ""
        _set_if_in_options(w_lora1, lora.get("slot1_file", "") or "")
        _set_if_in_options(w_lora2, lora.get("slot2_file", "") or "")
        _set_if_in_options(w_lora3, lora.get("slot3_file", "") or "")
        w1 = _safe_float(lora.get("slot1_weight", None), default=None)
        w2 = _safe_float(lora.get("slot2_weight", None), default=None)
        w3 = _safe_float(lora.get("slot3_weight", None), default=None)
        if w1 is not None: w_lora1_w.value = float(max(w_lora1_w.min, min(w_lora1_w.max, w1)))
        if w2 is not None: w_lora2_w.value = float(max(w_lora2_w.min, min(w_lora2_w.max, w2)))
        if w3 is not None: w_lora3_w.value = float(max(w_lora3_w.min, min(w_lora3_w.max, w3)))

def _copy_text_to_clipboard_js(text: str):
    from IPython.display import Javascript, display as _disp
    safe = json.dumps(str(text))
    _disp(Javascript(f"""
    (async () => {{ try {{ await navigator.clipboard.writeText({safe}); console.log("copied"); }} catch(e) {{ console.error(e); }} }})()
    """))

def _on_load_meta(_):
    out.clear_output()
    with out:
        png_path = (w_png_meta_path.value or "").strip()
        if not png_path: print("❌ 请先填写 PNG 路径"); return
        if not os.path.exists(png_path): print(f"❌ PNG 不存在: {png_path}"); return
        meta = _read_metadata_from_png(png_path)
        if not meta: print("❌ 未读到 parameters 元数据（该 PNG 可能不是本面板保存的，或没有 parameters 字段）"); return
        _apply_metadata_to_ui(meta)
        globals()["_last_loaded_png_metadata"] = meta
        meta_status.value = f"<div class='zimg-info'>✅ 已从 PNG 读取并回填：{os.path.basename(png_path)}</div>"
        print("✅ 元数据已回填到面板参数。")

def _on_copy_meta_json(_):
    meta = globals().get("_last_loaded_png_metadata", None)
    if not isinstance(meta, dict):
        meta_status.value = "<div class='zimg-info'>⚠️ 尚未读取任何 PNG 元数据；请先点击「一键读取」</div>"
        return
    text = json.dumps(meta, ensure_ascii=False, indent=2)
    _copy_text_to_clipboard_js(text)
    meta_status.value = "<div class='zimg-info'>📋 已将元数据 JSON 复制到剪贴板</div>"

btn_load_meta.on_click(_on_load_meta)
btn_copy_meta_json.on_click(_on_copy_meta_json)

_current_ratio = None
_updating = False

w_ratio = widgets.Dropdown(options=list(ASPECT_RATIOS.keys()), value="自定义", description="", layout=widgets.Layout(width="100%"))
w_width = widgets.IntSlider(value=1024, min=512, max=2560, step=16, description="宽度", style={'description_width': '60px'}, layout=widgets.Layout(width="100%"), readout=True, continuous_update=False)
w_height = widgets.IntSlider(value=1024, min=512, max=2560, step=16, description="高度", style={'description_width': '60px'}, layout=widgets.Layout(width="100%"), readout=True, continuous_update=False)
w_size_info = widgets.HTML('<div class="zimg-info">📐 最终尺寸: 1024 × 1024 (已对齐16像素)</div>')

def _parse_ratio_from_name(name):
    if name == "自定义": return None
    w, h = ASPECT_RATIOS[name]
    return w / h

def _set_initial_size_by_ratio(ratio):
    if ratio is None: return 1024, 1024
    if ratio >= 1: w = 1024; h = round(w / ratio)
    else: h = 1024; w = round(h * ratio)
    w = max(512, min(2560, _round_to_16(w)))
    h = max(512, min(2560, _round_to_16(h)))
    return w, h

def _on_ratio_change(change):
    global _current_ratio, _updating
    ratio_name = change["new"]
    ratio = _parse_ratio_from_name(ratio_name)
    _current_ratio = ratio
    if ratio is not None:
        w, h = _set_initial_size_by_ratio(ratio)
        _updating = True; w_width.value = w; w_height.value = h; _updating = False
        _update_size_info(None)
    else: _update_size_info(None)

def _on_width_change(change):
    global _current_ratio, _updating
    if _updating or _current_ratio is None: _update_size_info(change); return
    new_w = _round_to_16(change["new"])
    new_h = max(512, min(2560, _round_to_16(new_w / _current_ratio)))
    _updating = True; w_height.value = new_h; _updating = False
    _update_size_info(None)

def _on_height_change(change):
    global _current_ratio, _updating
    if _updating or _current_ratio is None: _update_size_info(change); return
    new_h = _round_to_16(change["new"])
    new_w = max(512, min(2560, _round_to_16(new_h * _current_ratio)))
    _updating = True; w_width.value = new_w; _updating = False
    _update_size_info(None)

def _update_size_info(change):
    w16 = _round_to_16(w_width.value); h16 = _round_to_16(w_height.value)
    w_size_info.value = f'<div class="zimg-info">📐 最终尺寸: {w16} × {h16} (已对齐16像素)</div>'

w_ratio.observe(_on_ratio_change, names="value")
w_width.observe(_on_width_change, names="value")
w_height.observe(_on_height_change, names="value")

w_steps = widgets.IntSlider(value=4, min=2, max=50, step=1, description="采样步数", style={'description_width': '80px'}, layout=widgets.Layout(width="100%"))
w_cfg = widgets.FloatSlider(value=1.0, min=0.0, max=10.0, step=0.1, description="CFG 强度", style={'description_width': '80px'}, layout=widgets.Layout(width="100%"))
w_seed = widgets.IntText(value=-1, description="随机种子", style={'description_width': '80px'}, layout=widgets.Layout(width="100%"))
w_batch = widgets.BoundedIntText(value=1, min=1, max=200, step=1, description="每次张数", style={'description_width': '80px'}, layout=widgets.Layout(width="220px"))

_SCHEDULER_OPTIONS =[("Euler（基线）", "euler"), ("Euler + Karras", "euler_karras"), ("Euler + Exponential", "euler_exp"), ("Euler Ancestral（stochastic）", "euler_ancestral"), ("Euler Ancestral + Karras", "euler_ancestral_karras"), ("DPM++ 2M RF", "dpm2m"), ("DPM++ 2M RF + Karras", "dpm2m_karras"), ("DPM++ 2M RF + Exponential", "dpm2m_exp")]
w_scheduler = widgets.Dropdown(options=_SCHEDULER_OPTIONS, value="euler", description="采样器", style={'description_width': '80px'}, layout=widgets.Layout(width="100%"))
w_shift = widgets.FloatSlider(value=3.0, min=0.5, max=10.0, step=0.1, description="shift", style={'description_width': '80px'}, layout=widgets.Layout(width="100%"), readout=True, continuous_update=False)

def _build_scheduler_from_ui(base_config: dict) -> object:
    from diffusers import FlowMatchEulerDiscreteScheduler
    try: from diffusers import FlowMatchDPMSolverMultistepScheduler; _has_dpm_rf = True
    except ImportError: _has_dpm_rf = False
    sel = w_scheduler.value; shift = float(w_shift.value)
    is_dpm = sel.startswith("dpm"); use_karras = sel.endswith("_karras"); use_exp = sel.endswith("_exp"); stochastic = "ancestral" in sel
    if is_dpm and not _has_dpm_rf:
        print("⚠️ 当前 diffusers 版本不含 FlowMatchDPMSolverMultistepScheduler，已回退到 Euler 基线")
        is_dpm = False
    _euler_supported = {"num_train_timesteps", "shift", "use_dynamic_shifting", "base_shift", "max_shift", "base_image_seq_len", "max_image_seq_len", "invert_sigmas", "use_karras_sigmas", "use_exponential_sigmas", "use_beta_sigmas", "time_shift_type", "stochastic_sampling", "shift_terminal"}
    cfg = dict(base_config)
    cfg["shift"] = shift; cfg["use_karras_sigmas"] = use_karras; cfg["use_exponential_sigmas"] = use_exp; cfg["stochastic_sampling"] = stochastic
    try:
        if is_dpm: return FlowMatchDPMSolverMultistepScheduler(**{k: v for k, v in cfg.items() if not k.startswith("_") and k != "stochastic_sampling"})
        else: return FlowMatchEulerDiscreteScheduler(**{k: v for k, v in cfg.items() if not k.startswith("_") and k in _euler_supported})
    except Exception as e: print(f"⚠️ Scheduler 构建失败，回退基线: {e}"); return None

w_attn = widgets.Dropdown(options=["auto", "max", 1, 2, 4, 8, 16, 32], value="auto", description="注意力切片", style={'description_width': '100px'}, layout=widgets.Layout(width="100%"))
w_int8 = widgets.Checkbox(value=True, description="启用 INT8 量化加速", style={'description_width': 'initial'}, layout=widgets.Layout(width="100%"))
w_dtype_mode = widgets.Dropdown(options=[("默认 FP32（基线）", "fp32"), ("省显存 FP16（推荐）", "fp16"), ("省显存 BF16（需GPU支持）", "bf16")], value="fp16", description="dtype", style={'description_width': '60px'}, layout=widgets.Layout(width="100%"))
w_dtype_apply = widgets.Checkbox(value=False, description="启用dtype下沉（transformer/TE 等尽力转为 FP16/BF16）", indent=False, layout=widgets.Layout(width="100%"))

# 💡 新增：大图防OOM极限模式开关
w_extreme_vram = widgets.Checkbox(value=False, description="🛡️ 极限大图防OOM (按步清显存 + CPU暴力解VAE，速度变慢)", style={'description_width': 'initial'}, layout=widgets.Layout(width="100%"))

_VAE_SCALING_FACTOR = 0.3611
w_tiled = widgets.Checkbox(value=False, description="启用 VAE 分块解码（大图推荐）", style={'description_width': 'initial'}, layout=widgets.Layout(width="100%"))
w_tile = widgets.Dropdown(options=["auto", 64, 128, 256, 384, 512, 768, 1024], value="auto", description="分块大小", style={'description_width': '80px'}, layout=widgets.Layout(width="100%"))

w_save_latent = widgets.Checkbox(value=False, description="保存 Latent (.npz)", style={'description_width': 'initial'}, layout=widgets.Layout(width="50%"))
w_save_png = widgets.Checkbox(value=False, description="保存 PNG 图像", style={'description_width': 'initial'}, layout=widgets.Layout(width="50%"))
w_latent_dir = widgets.Text(value="/content/drive/MyDrive/Z-image/Latent", description="Latent 目录", style={'description_width': '90px'}, layout=widgets.Layout(width="100%"))
w_png_dir = widgets.Text(value="/content/drive/MyDrive/Z-image/Outputs", description="PNG 目录", style={'description_width': '90px'}, layout=widgets.Layout(width="100%"))

_LORA_DIR = "/content/Lora"
_lora_merged_state: dict = {}
w_lora_dir = widgets.Text(value=_LORA_DIR, description="LoRA 目录", style={'description_width': '90px'}, layout=widgets.Layout(width="100%"))

def _list_lora_files_for_ui(folder: str):
    folder = (folder or "").strip()
    if not folder or not os.path.exists(folder): return []
    return sorted([fn for fn in os.listdir(folder) if fn.endswith(".safetensors")])

def _lora_load_sd(fp: str) -> dict:
    import safetensors.torch as sf; return sf.load_file(fp, device="cpu")

def _lora_get_transformer(p):
    t = getattr(p, "transformer", None)
    if t is None: raise RuntimeError("pipe.transformer 不存在")
    return t

def _lora_build_param_map(transformer) -> dict: return {k: v for k, v in transformer.named_parameters()}

def _lora_find_param(param_map: dict, base_key: str):
    for pfx in["diffusion_model.", "transformer.", "model.diffusion_model.", ""]:
        if base_key.startswith(pfx):
            bare = base_key[len(pfx):]
            if bare + ".weight" in param_map: return bare + ".weight", param_map[bare + ".weight"]
            if bare in param_map: return bare, param_map[bare]
    return None, None

def _lora_compute_delta(sd: dict, base_key: str, scale: float, param_device, param_dtype):
    key_A = base_key + ".lora_A.weight"; key_B = base_key + ".lora_B.weight"
    if key_A not in sd: key_A = base_key + ".lora_down.weight"; key_B = base_key + ".lora_up.weight"
    if key_A not in sd or key_B not in sd: return None
    A = sd[key_A].to(device=param_device, dtype=torch.float32)
    B = sd[key_B].to(device=param_device, dtype=torch.float32)
    return ((B @ A) * scale).to(dtype=param_dtype)

def _lora_merge_into_transformer(transformer, sd: dict, scale: float) -> int:
    param_map = _lora_build_param_map(transformer)
    base_keys = set(k[:-len(suffix)] for k in sd.keys() for suffix in[".lora_A.weight", ".lora_B.weight", ".lora_down.weight", ".lora_up.weight"] if k.endswith(suffix))
    merged = 0
    for base_key in base_keys:
        t_key, param = _lora_find_param(param_map, base_key)
        if param is None: continue
        delta = _lora_compute_delta(sd, base_key, scale, param.device, param.dtype)
        if delta is None or delta.shape != param.shape: continue
        param.data.add_(delta); merged += 1
    return merged

def _lora_unmerge_from_transformer(transformer, sd: dict, scale: float) -> int:
    param_map = _lora_build_param_map(transformer)
    base_keys = set(k[:-len(suffix)] for k in sd.keys() for suffix in[".lora_A.weight", ".lora_B.weight", ".lora_down.weight", ".lora_up.weight"] if k.endswith(suffix))
    unmerged = 0
    for base_key in base_keys:
        t_key, param = _lora_find_param(param_map, base_key)
        if param is None: continue
        delta = _lora_compute_delta(sd, base_key, scale, param.device, param.dtype)
        if delta is None or delta.shape != param.shape: continue
        param.data.sub_(delta); unmerged += 1
    return unmerged

def _apply_lora_slots_to_pipe(p, slot_files: list, slot_weights: list, lora_folder: str):
    global _lora_merged_state
    transformer = _lora_get_transformer(p)
    lora_folder = (lora_folder or "").strip()
    for i in range(1, 4):
        fn = slot_files[i - 1] or None
        weight = float(slot_weights[i - 1])
        cur = _lora_merged_state.get(i)
        if cur is not None and (cur["file"] != fn or abs(cur["weight"] - weight) > 1e-6):
            n = _lora_unmerge_from_transformer(transformer, cur["sd"], cur["weight"])
            print(f"   槽位{i} unmerge: {cur['file']} ({n} 层)")
            del _lora_merged_state[i]; cur = None
        if fn is not None and weight != 0.0 and cur is None:
            fp = os.path.join(lora_folder, fn)
            if not os.path.exists(fp): raise RuntimeError(f"LoRA 文件不存在: {fp}")
            sd = _lora_load_sd(fp)
            n = _lora_merge_into_transformer(transformer, sd, weight)
            if n == 0: print(f"   ⚠️ 槽位{i}: 未找到匹配层，LoRA 可能不兼容: {fn}")
            else: print(f"   槽位{i} merge: {fn} weight={weight} ({n} 层)")
            _lora_merged_state[i] = {"file": fn, "weight": weight, "sd": sd}

def _make_lora_slot_ui(slot_idx: int):
    dd = widgets.Dropdown(options=[("（不加载）", "")], value="", description=f"槽位{slot_idx}", layout=widgets.Layout(width="100%"))
    wt = widgets.FloatSlider(value=0.0, min=-2.0, max=2.0, step=0.05, description="强度", style={'description_width': '60px'}, layout=widgets.Layout(width="100%"), readout=True, continuous_update=False)
    return dd, wt

w_lora1, w_lora1_w = _make_lora_slot_ui(1)
w_lora2, w_lora2_w = _make_lora_slot_ui(2)
w_lora3, w_lora3_w = _make_lora_slot_ui(3)
btn_lora_refresh = widgets.Button(description="🔄 刷新 LoRA 文件列表", button_style="info", layout=widgets.Layout(width="220px", height="32px"))

def _refresh_lora_dropdowns(_=None):
    opts = [("（不加载）", "")] +[(fn, fn) for fn in _list_lora_files_for_ui(w_lora_dir.value.strip())]
    values = [v for _, v in opts]
    for w_lora in[w_lora1, w_lora2, w_lora3]:
        cur = w_lora.value
        w_lora.options = opts
        w_lora.value = cur if cur in values else ""

btn_lora_refresh.on_click(_refresh_lora_dropdowns)
w_lora_dir.observe(lambda c: _refresh_lora_dropdowns(), names="value")
_refresh_lora_dropdowns()

w_svh_enable = widgets.Checkbox(value=False, description="启用 SeedVarianceEnhancer", indent=False, layout=widgets.Layout(width="100%"))
w_svh_randomize_percent = widgets.FloatSlider(value=50.0, min=1.0, max=100.0, step=1.0, description="randomize%", style={'description_width': '110px'}, layout=widgets.Layout(width="100%"), readout=True, continuous_update=False)
w_svh_strength = widgets.FloatText(value=20.0, description="strength", style={'description_width': '110px'}, layout=widgets.Layout(width="100%"))
w_svh_noise_insert = widgets.Dropdown(options=["noise on beginning steps", "noise on ending steps", "noise on all steps", "disabled"], value="noise on all steps", description="noise_insert", style={'description_width': '110px'}, layout=widgets.Layout(width="100%"))
w_svh_steps_switchover_percent = widgets.FloatSlider(value=20.0, min=1.0, max=99.0, step=1.0, description="switch%", style={'description_width': '110px'}, layout=widgets.Layout(width="100%"), readout=True, continuous_update=False)
w_svh_seed_mode = widgets.Dropdown(options=[("跟随出图种子", "follow"), ("自定义 SVH 种子", "custom")], value="follow", description="SVH seed", style={'description_width': '110px'}, layout=widgets.Layout(width="100%"))
w_svh_seed = widgets.IntText(value=0, description="seed", style={'description_width': '110px'}, layout=widgets.Layout(width="100%"))
w_svh_mask_starts_at = widgets.Dropdown(options=["beginning", "end"], value="beginning", description="mask_from", style={'description_width': '110px'}, layout=widgets.Layout(width="100%"))
w_svh_mask_percent = widgets.FloatSlider(value=0.0, min=0.0, max=99.0, step=1.0, description="mask%", style={'description_width': '110px'}, layout=widgets.Layout(width="100%"), readout=True, continuous_update=False)
w_svh_log_to_console = widgets.Checkbox(value=False, description="log_to_console", indent=False, layout=widgets.Layout(width="100%"))

def _svh_controls_enabled(enable: bool):
    for w in[w_svh_randomize_percent, w_svh_strength, w_svh_noise_insert, w_svh_steps_switchover_percent, w_svh_seed_mode, w_svh_seed, w_svh_mask_starts_at, w_svh_mask_percent, w_svh_log_to_console]:
        w.disabled = not bool(enable)

def _on_svh_enable_change(change):
    _svh_controls_enabled(change["new"])
    _on_svh_seed_mode_change({"new": w_svh_seed_mode.value})

def _on_svh_seed_mode_change(change):
    w_svh_seed.disabled = (change["new"] != "custom") or (not bool(w_svh_enable.value))

w_svh_enable.observe(_on_svh_enable_change, names="value")
w_svh_seed_mode.observe(_on_svh_seed_mode_change, names="value")
_svh_controls_enabled(w_svh_enable.value)
_on_svh_seed_mode_change({"new": w_svh_seed_mode.value})

def _svh_apply_to_embeds(embeds_kwargs, *, enable, randomize_percent, strength, noise_insert, steps_switchover_percent, seed, mask_starts_at, mask_percent, log_to_console):
    if not enable or noise_insert == "disabled" or strength == 0: return embeds_kwargs
    if "prompt_embeds" not in embeds_kwargs or not isinstance(embeds_kwargs["prompt_embeds"], torch.Tensor): return embeds_kwargs
    rp = max(1.0, min(100.0, float(randomize_percent))) / 100.0
    mp = max(0.0, min(99.0, float(mask_percent))) / 100.0
    if noise_insert != "noise on all steps" and log_to_console: print(f"⚠️ SVH: 当前不支持按步段切换，已按 'noise on all steps' 处理。")
    def _first_null_last_nonnull(seq_t):
        first_null, last_nonnull = -1, -1
        null_seq = [0] * seq_t.size(1)
        for i in range(seq_t.size(1)):
            is_all_zero = bool(torch.all(seq_t[:, i, :] == 0).item())
            null_seq[i] = 0 if is_all_zero else 1
            if not is_all_zero: last_nonnull = i
            if is_all_zero and first_null == -1: first_null = i
        return first_null, last_nonnull, null_seq
    def _apply_one(t, local_seed):
        device = t.device
        torch.manual_seed(int(local_seed))
        noise = torch.rand_like(t) * 2 * float(strength) - float(strength)
        torch.manual_seed(int(local_seed) + 1)
        noise_mask = torch.bernoulli(torch.ones_like(t) * rp).bool()
        first_null, last_nonnull, null_seq = _first_null_last_nonnull(t)
        if mp > 0 or (last_nonnull < t.size(1) - 1 and last_nonnull >= 0):
            seq_len = (last_nonnull + 1) if (last_nonnull < t.size(1) - 1 and last_nonnull >= 0) else t.size(1)
            mask_start = (seq_len - int(seq_len * mp)) if mask_starts_at == "end" else 0
            mask_end = t.size(1) if mask_starts_at == "end" else int(seq_len * mp)
            prompt_mask = torch.arange(t.size(1), device=device).view(1, -1, 1).expand(t.size(0), -1, t.size(2))
            prompt_mask = (prompt_mask >= mask_start) & (prompt_mask < mask_end)
            if first_null > -1:
                null_mask_tensor = ~torch.tensor(null_seq, device=device, dtype=torch.bool).view(1, -1, 1).expand(t.size(0), -1, t.size(2))
                prompt_mask = prompt_mask | null_mask_tensor
            noise_mask = noise_mask & (~prompt_mask)
        return t + (noise * noise_mask)
    embeds_kwargs = dict(embeds_kwargs)
    embeds_kwargs["prompt_embeds"] = _apply_one(embeds_kwargs["prompt_embeds"], seed)
    if "negative_prompt_embeds" in embeds_kwargs and isinstance(embeds_kwargs["negative_prompt_embeds"], torch.Tensor):
        embeds_kwargs["negative_prompt_embeds"] = _apply_one(embeds_kwargs["negative_prompt_embeds"], seed + 999)
    return embeds_kwargs

btn_run = widgets.Button(description="🚀 生成图像", button_style="success", layout=widgets.Layout(width="160px", height="48px"))
btn_clear = widgets.Button(description="🗑️ 清空输出", button_style="warning", layout=widgets.Layout(width="140px", height="48px"))
out = widgets.Output(layout=widgets.Layout(border="2px solid #334155", border_radius="12px", padding="16px", min_height="300px", background="#0f172a"))

def _build_generation_metadata(prompt, negative, seed, steps, cfg, width, height, ratio_name, svh_meta, lora_slots, pipe=None, scheduler_name="", scheduler_shift=3.0):
    return {
        "prompt": prompt, "negative_prompt": negative, "seed": int(seed), "steps": int(steps), "cfg_scale": float(cfg),
        "width": int(width), "height": int(height), "aspect_ratio_preset": ratio_name, "scheduler": scheduler_name,
        "scheduler_shift": float(scheduler_shift), "lora_slots": lora_slots or {}, "seed_variance_enhancer": svh_meta or {},
        "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }

def _save_latent_with_metadata(latent, metadata, save_path):
    try:
        np.savez_compressed(save_path, latent=latent.detach().cpu().to(torch.float32).numpy(), metadata_json=np.array([json.dumps(metadata, ensure_ascii=False, indent=2)], dtype=object))
        return True
    except Exception as e: print(f"❌ 保存 Latent 失败: {e}"); traceback.print_exc(); return False

def _save_png_with_metadata(image, metadata, save_path):
    try:
        from PIL import PngImagePlugin
        pnginfo = PngImagePlugin.PngInfo()
        pnginfo.add_text("parameters", json.dumps(metadata, ensure_ascii=False, indent=2))
        for k in["prompt", "negative_prompt", "seed", "steps", "cfg_scale", "width", "height"]: pnginfo.add_text(k, str(metadata.get(k, "")))
        image.save(save_path, pnginfo=pnginfo)
        return True
    except Exception as e: print(f"❌ 保存 PNG 失败: {e}"); traceback.print_exc(); return False

# ========= 生成逻辑 =========
def _on_run(_):
    out.clear_output()
    with out:
        sdpa_ctx = None
        _original_scheduler = None
        try:
            p = _get_existing_pipe()
            if p is None:
                print("❌ 未检测到 pipe：请先用模型管理器加载模型")
                return

            prompt      = prompt_box.value.strip()
            negative    = neg_box.value.strip()
            width       = int(w_width.value)
            height      = int(w_height.value)
            steps       = int(w_steps.value)
            cfg         = float(w_cfg.value)
            seed_in     = int(w_seed.value)
            batch_n     = int(w_batch.value)

            attn_slice  = w_attn.value
            int8_enable = bool(w_int8.value)
            scaling     = _VAE_SCALING_FACTOR
            tiled       = bool(w_tiled.value)
            tile_size   = w_tile.value

            # 💡 读取极限防OOM模式开关
            extreme_vram_mode = bool(w_extreme_vram.value)

            save_latent = bool(w_save_latent.value)
            save_png    = bool(w_save_png.value)
            latent_dir  = w_latent_dir.value.strip()
            png_dir     = w_png_dir.value.strip()

            os.makedirs(latent_dir, exist_ok=True)
            os.makedirs(png_dir, exist_ok=True)

            width16  = _round_to_16(width)
            height16 = _round_to_16(height)
            if tile_size == "auto": tile_size = _auto_tile_for_size(width16, height16)

            dtype_mode   = str(w_dtype_mode.value)
            target_dtype = _dtype_from_mode(dtype_mode)
            if target_dtype == torch.bfloat16 and not _bf16_is_supported_cuda():
                print("⚠️ BF16 似乎不支持，自动降级为 FP16")
                target_dtype = torch.float16
            dtype_apply_enabled = bool(w_dtype_apply.value)

            svh_enabled  = bool(w_svh_enable.value)
            svh_seed_mode = str(w_svh_seed_mode.value)

            _original_scheduler = p.scheduler
            _new_sched = _build_scheduler_from_ui(dict(p.scheduler.config))
            if _new_sched is not None:
                p.scheduler = _new_sched
                sched_label = w_scheduler.value
            else:
                sched_label = "euler（回退基线）"

            print("=" * 50)
            print(f"🎯 批量生成: {batch_n} 张")
            print(f"   尺寸: {width16}×{height16} | 步数: {steps} | CFG: {cfg}")
            print(f"   极限大图防OOM模式: {'✅ 开启' if extreme_vram_mode else '❌ 关闭'}")
            print(f"   scheduler: {sched_label} | shift: {w_shift.value}")
            print("=" * 50)

            if torch.cuda.is_available():
                try: torch.cuda.reset_peak_memory_stats()
                except Exception: pass

            _prepare_full_gpu(p, attn_slice)
            _maybe_cast_pipe_modules(p, target_dtype=target_dtype, enable=dtype_apply_enabled)

            sdpa_ctx = _enter_sdpa_efficient()

            ok_int8, msg_int8 = _apply_int8_matmul_to_transformer(p, int8_enable)
            if not ok_int8: print("⚠️ " + msg_int8)

            slot_files   =[w_lora1.value or None, w_lora2.value or None, w_lora3.value or None]
            slot_weights =[float(w_lora1_w.value), float(w_lora2_w.value), float(w_lora3_w.value)]
            lora_folder  = w_lora_dir.value.strip()

            try:
                _apply_lora_slots_to_pipe(p, slot_files=slot_files, slot_weights=slot_weights, lora_folder=lora_folder)
            except Exception as e:
                print("⚠️ LoRA 应用失败（已忽略，不影响无LoRA推理）:", e)

            lora_slots_meta = {"slot1_file": slot_files[0] or "", "slot1_weight": slot_weights[0], "slot2_file": slot_files[1] or "", "slot2_weight": slot_weights[1], "slot3_file": slot_files[2] or "", "slot3_weight": slot_weights[2], "lora_folder": lora_folder}

            for i in range(batch_n):
                result = None; latent = None; img = None; embeds_kwargs = None; metadata = None; gen = None
                try:
                    real_seed = int(np.random.randint(0, 2**32 - 1)) if seed_in == -1 else int(seed_in + i)
                    svh_seed  = int(w_svh_seed.value) if (svh_seed_mode == "custom") else int(real_seed)
                    svh_meta = {"enabled": svh_enabled, "randomize_percent": float(w_svh_randomize_percent.value), "strength": float(w_svh_strength.value), "noise_insert": str(w_svh_noise_insert.value), "steps_switchover_percent": float(w_svh_steps_switchover_percent.value), "seed_mode": str(w_svh_seed_mode.value), "seed": int(svh_seed), "mask_starts_at": str(w_svh_mask_starts_at.value), "mask_percent": float(w_svh_mask_percent.value)}

                    print("-" * 50)
                    print(f"🖼️ [{i+1}/{batch_n}] seed={real_seed}")

                    t0 = time.time()
                    with torch.inference_mode():
                        te_dtype = target_dtype if dtype_apply_enabled else torch.float16
                        embeds_kwargs = _encode_prompt_embeds_then_drop_te(p, prompt_text=prompt, negative_text=negative, cfg=cfg, te_dtype=te_dtype)
                        embeds_kwargs = _svh_apply_to_embeds(embeds_kwargs, enable=svh_enabled, randomize_percent=float(w_svh_randomize_percent.value), strength=float(w_svh_strength.value), noise_insert=str(w_svh_noise_insert.value), steps_switchover_percent=float(w_svh_steps_switchover_percent.value), seed=int(svh_seed), mask_starts_at=str(w_svh_mask_starts_at.value), mask_percent=float(w_svh_mask_percent.value), log_to_console=bool(w_svh_log_to_console.value))

                        if torch.cuda.is_available(): runtime_fragmentation_cleanup(synchronize=True, reset_peak=True)

                        gen = torch.Generator(device="cpu").manual_seed(real_seed)

                        # 构建基础参数
                        pipe_kwargs = {
                            "prompt": None,
                            **embeds_kwargs,
                            "height": height16,
                            "width": width16,
                            "num_inference_steps": steps,
                            "guidance_scale": cfg,
                            "generator": gen,
                            "output_type": "latent"
                        }

                        # 💡 如果开启了极限防爆模式，装载步间清道夫
                        if extreme_vram_mode:
                            def step_vram_cleaner(pipe, step_index, timestep, callback_kwargs):
                                alloc = torch.cuda.memory_allocated() / (1024**3)
                                # 设置警戒线为 13.5GB，超过立刻粉碎显存碎片
                                if alloc > 13.5:
                                    torch.cuda.empty_cache()
                                    torch.cuda.ipc_collect()
                                return callback_kwargs
                            pipe_kwargs["callback_on_step_end"] = step_vram_cleaner

                        # 执行生图
                        result = p(**pipe_kwargs)
                        latent = result.images[0].clone()

                    # 💡 将 extreme_vram_mode 标志传给解码器，如果是 True 则启用 CPU 解码
                    img = _decode_latent_with_pipe_vae(p, latent, scaling_factor=scaling, tiled=tiled, tile_size=tile_size, extreme_vram_mode=extreme_vram_mode)
                    display(img)

                    metadata = _build_generation_metadata(prompt=prompt, negative=negative, seed=real_seed, steps=steps, cfg=cfg, width=width16, height=height16, ratio_name=w_ratio.value, svh_meta=svh_meta, lora_slots=lora_slots_meta, pipe=p, scheduler_name=sched_label, scheduler_shift=float(w_shift.value))
                    ts = datetime.now().strftime("%H%M%S")
                    base_name = f"{real_seed}_{width16}x{height16}_{steps}s_{ts}"

                    if save_latent:
                        latent_path = os.path.join(latent_dir, f"latent_{base_name}.npz")
                        if _save_latent_with_metadata(latent, metadata, latent_path): print(f"💾 Latent: {latent_path}")
                    if save_png:
                        png_path = os.path.join(png_dir, f"img_{base_name}.png")
                        if _save_png_with_metadata(img, metadata, png_path): print(f"💾 PNG: {png_path}")

                    dt = time.time() - t0
                    print(f"⏱️ 单张耗时: {dt:.2f} 秒")

                except KeyboardInterrupt:
                    print(f"\n🛑 图像[{i+1}/{batch_n}] 已手动中断！")
                    raise
                except Exception:
                    traceback.print_exc()
                finally:
                    if 'result' in locals(): del result
                    if 'latent' in locals(): del latent
                    if 'img' in locals(): del img
                    if 'embeds_kwargs' in locals(): del embeds_kwargs
                    if 'metadata' in locals(): del metadata
                    if 'gen' in locals(): del gen

                    gc.collect()
                    if torch.cuda.is_available():
                        try: torch.cuda.synchronize()
                        except Exception: pass
                        try: torch.cuda.empty_cache()
                        except Exception: pass
                        try: torch.cuda.ipc_collect()
                        except Exception: pass

            print("=" * 50)
            print("✨ 批量完成！")

        except KeyboardInterrupt:
            print("\n🛑 批量生成流程已手动中止！")
        except Exception:
            traceback.print_exc()
        finally:
            try:
                if p is not None and _original_scheduler is not None:
                    p.scheduler = _original_scheduler
            except Exception: pass

            _exit_sdpa(sdpa_ctx)

            try:
                if p is not None:
                    _, _, drop_te_fn = _require_manager_funcs()
                    drop_te_fn(p, drop_tokenizer=False)
            except Exception: pass

            gc.collect()
            if torch.cuda.is_available():
                try: torch.cuda.synchronize()
                except Exception: pass
                try: torch.cuda.empty_cache()
                except Exception: pass
                try: torch.cuda.ipc_collect()
                except Exception: pass

            if torch.cuda.is_available():
                alloc = torch.cuda.memory_allocated() / (1024**3)
                resv  = torch.cuda.memory_reserved() / (1024**3)
                print(f"🏁 运行结束 (含清理)，当前显存: 占用 {alloc:.2f} GB | 保留 {resv:.2f} GB")

def _on_clear(_):
    out.clear_output()

_row = lambda *ws, **kw: widgets.HBox(list(ws), layout=widgets.Layout(align_items="center", gap="8px", width="100%", **kw))

def make_section(title_html, *children):
    return widgets.VBox([widgets.HTML(f'<div class="zimg-section-title">{title_html}</div>'), *children], layout=widgets.Layout(padding="10px 14px", margin="6px 0", border="1px solid #334155", border_radius="10px", background="#1e293b"))

def make_collapsible_section(title_html, content_widget, open_default=True):
    acc = widgets.Accordion(children=[content_widget])
    acc.set_title(0, title_html)
    acc.selected_index = 0 if open_default else None
    acc.layout = widgets.Layout(margin="5px 0")
    return acc

prompt_box.layout.height = "220px"; neg_box.layout.height = "44px"
prompt_inner = widgets.VBox([widgets.HTML('<div class="zimg-label">正向提示词</div>'), prompt_box, widgets.HTML('<div class="zimg-label" style="margin-top:6px;">负面提示词（CFG≤1无效）</div>'), neg_box, widgets.HTML('<div class="zimg-label" style="margin-top:6px;">📥 复制/回填 PNG 元数据</div>'), w_png_meta_path, _row(btn_load_meta, btn_copy_meta_json), meta_status], layout=widgets.Layout(padding="10px 14px"))
prompt_section = make_collapsible_section("📝 提示词", prompt_inner, open_default=True)

w_width.layout  = widgets.Layout(flex="1", min_width="0"); w_height.layout = widgets.Layout(flex="1", min_width="0")
w_width.style   = {'description_width': '36px'}; w_height.style  = {'description_width': '36px'}
w_steps.layout = widgets.Layout(flex="1", min_width="0"); w_cfg.layout   = widgets.Layout(flex="1", min_width="0")
w_steps.style  = {'description_width': '60px'}; w_cfg.style    = {'description_width': '60px'}
w_seed.layout      = widgets.Layout(flex="1", min_width="0", width="auto"); w_scheduler.layout = widgets.Layout(flex="2", min_width="0", width="auto")
w_seed.style       = {'description_width': '60px'}; w_scheduler.style  = {'description_width': '60px'}
w_shift.layout = widgets.Layout(width="100%"); w_shift.style  = {'description_width': '60px'}
w_lora_dir.layout = widgets.Layout(flex="1", min_width="0", width="auto"); w_lora_dir.style  = {'description_width': '70px'}
w_lora1.layout   = widgets.Layout(flex="2", min_width="0", width="auto"); w_lora1_w.layout = widgets.Layout(flex="1", min_width="0", width="auto"); w_lora1_w.style  = {'description_width': '40px'}
w_lora2.layout   = widgets.Layout(flex="2", min_width="0", width="auto"); w_lora2_w.layout = widgets.Layout(flex="1", min_width="0", width="auto"); w_lora2_w.style  = {'description_width': '40px'}
w_lora3.layout   = widgets.Layout(flex="2", min_width="0", width="auto"); w_lora3_w.layout = widgets.Layout(flex="1", min_width="0", width="auto"); w_lora3_w.style  = {'description_width': '40px'}
lora_slots_ui_compact = widgets.VBox([_row(w_lora_dir, btn_lora_refresh), widgets.HTML("<div class='zimg-info'>⚠️ 加载 LoRA 可能导致 2048² 显存不够；可降分辨率。</div>"), _row(w_lora1, w_lora1_w), _row(w_lora2, w_lora2_w), _row(w_lora3, w_lora3_w)])
w_latent_dir.layout = widgets.Layout(flex="1", min_width="0", width="auto"); w_png_dir.layout    = widgets.Layout(flex="1", min_width="0", width="auto")
w_latent_dir.style  = {'description_width': '70px'}; w_png_dir.style     = {'description_width': '70px'}
w_attn.layout       = widgets.Layout(flex="1", min_width="0", width="auto"); w_dtype_mode.layout = widgets.Layout(flex="0.8", min_width="0", width="auto")
w_attn.style        = {'description_width': '70px'}; w_dtype_mode.style  = {'description_width': '50px'}
w_int8.layout       = widgets.Layout(flex="1", min_width="0", width="auto"); w_dtype_apply.layout= widgets.Layout(flex="1", min_width="0", width="auto")
w_tile.layout  = widgets.Layout(flex="1", min_width="0", width="auto"); w_tile.style   = {'description_width': '60px'}
w_tiled.layout = widgets.Layout(flex="1", min_width="0", width="auto")

combined_ops_content = widgets.VBox([
    widgets.HTML('<div class="zimg-section-title" style="margin-bottom:4px;">📐 图像尺寸</div>'), w_ratio, _row(w_width, w_height), w_size_info,
    widgets.HTML('<hr style="border-color:#334155;margin:8px 0;">'),
    widgets.HTML('<div class="zimg-section-title" style="margin-bottom:4px;">⚙️ 生成参数 / 显存</div>'), _row(w_steps, w_cfg), _row(w_seed, w_scheduler), w_shift, _row(w_attn, w_int8), _row(w_dtype_mode, w_dtype_apply, layout=widgets.Layout(align_items="center", gap="12px")),
    # 💡 在此处加入了防 OOM 开关
    _row(w_extreme_vram),
    widgets.HTML('<hr style="border-color:#334155;margin:8px 0;">'),
    widgets.HTML('<div class="zimg-section-title" style="margin-bottom:4px;">🖼️ VAE 解码</div>'), _row(w_tiled, w_tile),
    widgets.HTML('<hr style="border-color:#334155;margin:8px 0;">'),
    widgets.HTML('<div class="zimg-section-title" style="margin-bottom:4px;">🎭 LoRA（3槽位·推理前自动应用）</div>'), lora_slots_ui_compact,
    widgets.HTML('<hr style="border-color:#334155;margin:8px 0;">'),
    widgets.HTML('<div class="zimg-section-title" style="margin-bottom:4px;">💾 保存设置</div>'), _row(w_save_latent, w_save_png), _row(w_latent_dir, w_png_dir),
], layout=widgets.Layout(padding="10px 14px"))
combined_ops_section = make_collapsible_section("⚙️ 图像尺寸 / 生成参数 / LoRA / 保存", combined_ops_content, open_default=True)

for _w, _flex in[(w_svh_randomize_percent, "1"), (w_svh_strength, "1"), (w_svh_noise_insert, "2"), (w_svh_steps_switchover_percent, "1"), (w_svh_mask_starts_at, "1"), (w_svh_mask_percent, "1"), (w_svh_seed_mode, "2"), (w_svh_seed, "1")]:
    _w.layout = widgets.Layout(flex=_flex, min_width="0", width="auto")
    if hasattr(_w, 'style'): _w.style = {'description_width': '80px'}

svh_section_content = make_section("🧬 SeedVarianceEnhancer", widgets.HTML("""<div style="color:#64748b;font-size:11px;padding:6px 8px;background:#0f172a;border-radius:6px;line-height:1.5;">TE 编码后对 <code>prompt_embeds</code>（CFG&gt;1 含 negative）做噪声注入，增强多样性。<br>⚠️ Colab 不支持按步 begin/end 切换，begin/end 等价 all steps 处理。</div>"""), w_svh_enable, _row(w_svh_randomize_percent, w_svh_strength), _row(w_svh_noise_insert, w_svh_steps_switchover_percent), _row(w_svh_mask_starts_at, w_svh_mask_percent), _row(w_svh_seed_mode, w_svh_seed), w_svh_log_to_console)
svh_section = make_collapsible_section("🧬 SeedVarianceEnhancer（多样性/种子扰动）", svh_section_content, open_default=False)

btn_run.on_click(_on_run)
btn_clear.on_click(_on_clear)

button_section = widgets.HBox([btn_run, w_batch, btn_clear], layout=widgets.Layout(justify_content="center", margin="12px 0", gap="16px", align_items="center"))

main_panel = widgets.VBox([header_html, allocator_html, prompt_section, combined_ops_section, svh_section, button_section, widgets.HTML('<div class="zimg-section-title" style="margin-top:12px;">📺 输出区域</div>'), out], layout=widgets.Layout(width="800px", padding="16px", border_radius="16px", background="#0f172a"))

display(main_panel)

📖 使用说明：

1，画面效果最好：完整Turbo INT8

2，自由度最高：UINT4+Base INT8，速度最慢，需20步以上出图，但可用base模型常规CFG，负面提示词也生效。

3，最快出图：UINT4+Base INT8+融合distill lora，根据融合的lora，4-16步出图，steps原则是lora文件名“steps”前面数字x1-2，推理步数过多反而不好，自行在清晰和油腻之间找到甜点区。

4，Base INT8目前暂时仅有基础的Euler可用，其他还在摸索调试。

5，若要使用Base INT8模型，为搭配UINT4 TE必须首先进行base模型定制量化，进行量化时连接CPU，耗时越10-15分钟，量化结束后断开连接，重新连接GPU,量化只进行一次，结果存放在google driver，今后再使用，不需要再量化。

6，混搭Base INT8，可选择“缓存到本地”，提高后续加载速度。

7，Tubor模型无需叠加使用Lora-Distill，CFG永远置于0！


